# 04_gold
Construye las tablas gold a partir de las tablas silver y de las tablas de clasificación `bronze_clubes_externos` y `bronze_competiciones`. Cada tabla se reescribe completa (`overwrite`) en `workspace.inferiores`. Las celdas dependen unas de otras: hay que correrlas en el orden del notebook.

Control del ciclo: `%run ./ciclo` y `motivo_no_corre("publicar")` al principio, y `marcar("publicado_en")` al final.

## Tablas, en orden de construcción

| Tabla | Una fila por |
|---|---|
| `gold_club_familia` | club (padre, juvenil o marcador `no_club`) |
| `gold_linea_tiempo` | jugador y paso por un club (tramo) |
| `gold_debut` | jugador |
| `gold_formacion_fifa` | jugador, año calendario y club familia |
| `gold_formacion_debut` | jugador, año calendario y club familia, cortado en el debut |
| `gold_ranking_formados` | club familia |
| `gold_salidas` | salida de un jugador desde un club juvenil |
| `gold_ranking_salidos` | club familia |
| `gold_ranking_camada` | club familia y camada |
| `gold_jugador` | jugador |
| `gold_seleccion` | jugador con al menos una selección |
| `gold_valor_mercado_serie` | punto de valor de mercado |
| `gold_valor_mercado` | jugador |
| `gold_minutos` | jugador, temporada, competición y club |
| `gold_logros` | detalle de cada logro |
| `gold_competicion` | competición |
| `gold_aporte` | jugador y club formador |
| `gold_ingresos` | traspaso y club (venta propia o solidaridad) |

## Términos

| Término | Significado |
|---|---|
| Club familia | Club padre al que pertenece un club: un juvenil pertenece a su padre. Un club sin fila en `gold_club_familia` es su propia familia |
| Camada | Año de debut del jugador en un club que no es juvenil (`temporada_debut` de `gold_debut`) |
| Relleno | Parte del peso de formación que se asigna hacia atrás, desde el primer club que figura en la carrera del jugador, para completar los años de los 12 a los 23. Las columnas `*_sin_relleno` lo descuentan |
| `es_argentino` en un jugador | Tiene ciudadanía argentina (`silver_player_citizenship`) |
| `es_argentino` en un club | Su país más frecuente en `silver_transfers` es Argentina |
| `en_los_105` | El club es un club padre argentino de `bronze_youth_teams_v2` (`origen = argentino_105`) |

Después de escribir cada tabla, la celda ejecuta consultas de control (conteos, claves únicas, comparaciones contra silver y muestras al azar).

In [0]:
%run ./ciclo

In [0]:
motivo_salida = motivo_no_corre("publicar")
if motivo_salida:
    dbutils.notebook.exit("SALTEADO: " + motivo_salida)

## gold_club_familia
Una fila por club. Reúne los clubes padre argentinos de `bronze_youth_teams_v2` y sus equipos juveniles (`origen = argentino_105`) y los clubes clasificados por `clasificar_clubes_externos` (`origen = externo`).

- El nombre de cada club padre argentino es el más frecuente en `silver_transfers` (origen y destino). Si no figura, se usa el `club_slug`.
- De `bronze_clubes_externos` entran los roles `padre`, `juvenil` y `no_club`, con una fila por club (la visita más antigua) y sin los clubes que ya están en `bronze_youth_teams_v2`.
- `parent_club_id`: el propio club para un padre, el club padre para un juvenil y nulo para un `no_club`.

Columnas: `club_id`, `parent_club_id`, `rol`, `tipo`, `club_name` y `origen`.

Controles: tamaño y unicidad de `club_id`, desglose por `origen` y `rol`, padres argentinos que quedaron con el slug como nombre y juveniles cuyo padre no existe en la tabla.

In [0]:
# EN LIMPIO: gold_club_familia ampliada (escribe workspace.inferiores.gold_club_familia)
# 105 clubes argentinos + 215 juveniles (bronze_youth_teams_v2) + clubes clasificados por visita (bronze_clubes_externos)
T = "workspace.inferiores"

# 1) Nombre de cada club padre argentino: el más frecuente en silver_transfers (origen y destino)
spark.sql(f"""
WITH usos AS (
  SELECT from_club_id AS club_id, from_club_name AS nombre FROM {T}.silver_transfers
  UNION ALL
  SELECT to_club_id, to_club_name FROM {T}.silver_transfers
),
cnt AS (
  SELECT club_id, nombre, COUNT(*) AS n
  FROM usos
  WHERE nombre IS NOT NULL
    AND club_id IN (SELECT DISTINCT CAST(club_id AS STRING) FROM {T}.bronze_youth_teams_v2)
  GROUP BY club_id, nombre
)
SELECT club_id, nombre
FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY club_id ORDER BY n DESC, nombre) AS rn FROM cnt)
WHERE rn = 1
""").createOrReplaceTempView("nombres_padre")

# 2) Tabla: argentinos (105 padres + 215 juveniles) + externos (padre, juvenil y marcadores no_club)
spark.sql(f"""
SELECT CAST(team_id AS STRING) AS club_id,
       CAST(club_id AS STRING) AS parent_club_id,
       'juvenil'               AS rol,
       trim(tipo)              AS tipo,
       trim(nombre)            AS club_name,
       'argentino_105'         AS origen
FROM {T}.bronze_youth_teams_v2

UNION ALL
SELECT p.club_id,
       p.club_id               AS parent_club_id,
       'padre'                 AS rol,
       CAST(NULL AS STRING)    AS tipo,
       COALESCE(n.nombre, p.club_slug) AS club_name,
       'argentino_105'         AS origen
FROM (SELECT DISTINCT CAST(club_id AS STRING) AS club_id, club_slug
      FROM {T}.bronze_youth_teams_v2) p
LEFT JOIN nombres_padre n ON p.club_id = n.club_id

UNION ALL
SELECT CAST(e.club_id AS STRING) AS club_id,
       CASE WHEN e.rol = 'padre'   THEN CAST(e.club_id AS STRING)
            WHEN e.rol = 'juvenil' THEN NULLIF(trim(CAST(e.parent_club_id AS STRING)), '')
       END                       AS parent_club_id,
       e.rol                     AS rol,
       CAST(NULL AS STRING)      AS tipo,
       trim(e.nombre)            AS club_name,
       'externo'                 AS origen
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY CAST(club_id AS STRING)
                               ORDER BY visitado_en, CAST(parent_club_id AS STRING)) AS rn
  FROM {T}.bronze_clubes_externos
  WHERE rol IN ('padre', 'juvenil', 'no_club')
) e
WHERE e.rn = 1
  AND CAST(e.club_id AS STRING) NOT IN (
        SELECT CAST(team_id AS STRING) FROM {T}.bronze_youth_teams_v2
        UNION
        SELECT CAST(club_id AS STRING) FROM {T}.bronze_youth_teams_v2)
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_club_familia")

# A: tamaño y unicidad (filas debe ser igual a ids_distintos)
display(spark.sql(f"""
SELECT COUNT(*) AS filas, COUNT(DISTINCT club_id) AS ids_distintos
FROM {T}.gold_club_familia"""))

# B: desglose por origen y rol
display(spark.sql(f"""
SELECT origen, rol, COUNT(*) AS filas
FROM {T}.gold_club_familia
GROUP BY origen, rol
ORDER BY origen, rol"""))

# C: padres argentinos que quedaron con el slug como nombre
display(spark.sql(f"""
SELECT COUNT(*) AS padres_sin_nombre_en_traspasos
FROM (SELECT DISTINCT CAST(club_id AS STRING) AS club_id FROM {T}.bronze_youth_teams_v2) p
LEFT JOIN nombres_padre n ON p.club_id = n.club_id
WHERE n.nombre IS NULL"""))

# D: juveniles cuyo padre no existe en la tabla
display(spark.sql(f"""
SELECT j.club_id, j.club_name, j.parent_club_id, j.origen
FROM {T}.gold_club_familia j
LEFT JOIN {T}.gold_club_familia p ON j.parent_club_id = p.club_id AND p.rol = 'padre'
WHERE j.rol = 'juvenil' AND p.club_id IS NULL"""))

## gold_linea_tiempo
Una fila por jugador y paso por un club (`tramo`), del más viejo al más nuevo. Excluye los traspasos futuros (`upcoming`).

- **Tramos:** el tramo 1 es el club de origen del traspaso más viejo (`tramo_inicial`, sin `fecha_desde`). Después hay un tramo por traspaso: el club de destino desde la fecha del traspaso hasta la del siguiente. `tramo_actual` marca el último.
- **Cesiones:** `esta_cedido` es verdadero cuando el traspaso es de tipo `loan`; en ese caso el club dueño es el de origen. Si no, el dueño es el club de destino.
- **Fechas sin dato:** se estiman con la temporada (1 de julio del año en que empieza) y se acotan con las fechas reales vecinas, para que no queden fuera de orden. `*_aprox` indica que la fecha se estimó y `*_ajustada` que, además, se corrió respecto de la estimación inicial.

Columnas principales: `player_id`, `tramo`, `club_id`, `club_name`, `rol_club` (`adulto` si el club no está en `gold_club_familia`), `tipo_club`, `club_familia_id`, `club_dueno_id`, `familia_dueno_id`, `esta_cedido`, `tipo_entrada`, `fecha_desde`, `fecha_hasta`, `edad_desde`, `edad_hasta`, `tramo_inicial` y `tramo_actual`.

Controles: cantidad de tramos y jugadores, tramos con `fecha_hasta` anterior a `fecha_desde`, tramos adultos anteriores al primer padre y cantidad de fechas aproximadas y ajustadas.

In [0]:
# EN LIMPIO: gold_linea_tiempo (escribe workspace.inferiores.gold_linea_tiempo)
# Una fila por jugador y paso por un club, del más viejo al más nuevo.
# Fechas sin dato: se estiman con la temporada, acotadas por las fechas reales vecinas.
T = "workspace.inferiores"

spark.sql(f"""
WITH base AS (
  SELECT CAST(player_id AS STRING) AS player_id,
         orden, transfer_type, age,
         CAST(from_club_id AS STRING) AS from_id, from_club_name AS from_name,
         CAST(to_club_id   AS STRING) AS to_id,   to_club_name   AS to_name,
         date AS fecha_real,
         COALESCE(date, make_date(season_start_year, 7, 1)) AS fecha_prop,
         ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY orden DESC) AS k
  FROM {T}.silver_transfers
  WHERE COALESCE(upcoming, false) = false
),
aj AS (
  SELECT *,
         MAX(fecha_prop) OVER (PARTITION BY player_id ORDER BY k
                               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS max_previo,
         MIN(fecha_real) OVER (PARTITION BY player_id ORDER BY k
                               ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) AS min_real_sig
  FROM base
),
ef AS (
  SELECT *,
         CASE WHEN fecha_real IS NOT NULL THEN fecha_real
              ELSE LEAST(max_previo, COALESCE(min_real_sig, max_previo)) END AS fecha,
         (fecha_real IS NULL) AS aprox
  FROM aj
),
t AS (
  SELECT *,
         (aprox AND fecha <> fecha_prop) AS ajustada,
         LEAD(fecha) OVER (PARTITION BY player_id ORDER BY k) AS fecha_sig,
         LEAD(aprox) OVER (PARTITION BY player_id ORDER BY k) AS aprox_sig,
         LEAD(aprox AND fecha <> fecha_prop) OVER (PARTITION BY player_id ORDER BY k) AS ajustada_sig,
         LEAD(age)   OVER (PARTITION BY player_id ORDER BY k) AS age_sig
  FROM ef
),
tramos AS (
  -- tramo inicial: el club de origen del traspaso más viejo
  SELECT player_id, 1 AS tramo,
         from_id AS club_id, from_name AS club_name, from_id AS dueno_id,
         false AS esta_cedido, CAST(NULL AS STRING) AS tipo_entrada,
         CAST(NULL AS DATE) AS fecha_desde, false AS desde_aprox, false AS desde_ajustada, CAST(NULL AS INT) AS edad_desde,
         fecha AS fecha_hasta, aprox AS hasta_aprox, ajustada AS hasta_ajustada, CAST(age AS INT) AS edad_hasta,
         true AS tramo_inicial
  FROM t WHERE k = 1
  UNION ALL
  -- un tramo por traspaso: el club de destino hasta el traspaso siguiente
  SELECT player_id, k + 1,
         to_id, to_name,
         CASE WHEN transfer_type = 'loan' THEN from_id ELSE to_id END,
         (transfer_type = 'loan'), transfer_type,
         fecha, aprox, ajustada, CAST(age AS INT),
         fecha_sig, aprox_sig, ajustada_sig, CAST(age_sig AS INT),
         false
  FROM t
)
SELECT s.player_id, s.tramo,
       s.club_id, s.club_name,
       COALESCE(c.rol, 'adulto')               AS rol_club,
       c.tipo                                  AS tipo_club,
       COALESCE(c.parent_club_id, s.club_id)   AS club_familia_id,
       s.dueno_id                              AS club_dueno_id,
       COALESCE(d.parent_club_id, s.dueno_id)  AS familia_dueno_id,
       s.esta_cedido, s.tipo_entrada,
       s.fecha_desde, s.desde_aprox AS fecha_desde_aprox, s.desde_ajustada AS fecha_desde_ajustada, s.edad_desde,
       s.fecha_hasta, s.hasta_aprox AS fecha_hasta_aprox, s.hasta_ajustada AS fecha_hasta_ajustada, s.edad_hasta,
       s.tramo_inicial,
       (s.tramo = MAX(s.tramo) OVER (PARTITION BY s.player_id)) AS tramo_actual
FROM tramos s
LEFT JOIN {T}.gold_club_familia c ON s.club_id  = c.club_id
LEFT JOIN {T}.gold_club_familia d ON s.dueno_id = d.club_id
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_linea_tiempo")

# A: tamaño y jugadores (jugadores = los que tienen algún traspaso que no es upcoming)
display(spark.sql(f"""
SELECT (SELECT COUNT(*) FROM {T}.gold_linea_tiempo) AS tramos,
       (SELECT COUNT(DISTINCT player_id) FROM {T}.gold_linea_tiempo) AS jugadores_en_linea,
       (SELECT COUNT(DISTINCT player_id) FROM {T}.silver_transfers WHERE COALESCE(upcoming, false) = false) AS jugadores_con_pasado_en_silver"""))

# B: tramos con fecha_hasta anterior a fecha_desde (esperado: 0)
display(spark.sql(f"""
SELECT COUNT(*) AS hasta_antes_que_desde
FROM {T}.gold_linea_tiempo
WHERE fecha_desde IS NOT NULL AND fecha_hasta IS NOT NULL AND fecha_hasta < fecha_desde"""))

# C: tramos adultos anteriores al primer padre (esperado: 0)
display(spark.sql(f"""
WITH pp AS (
  SELECT player_id, MIN(tramo) AS tramo_primer_padre
  FROM {T}.gold_linea_tiempo WHERE rol_club = 'padre' GROUP BY player_id
)
SELECT COUNT(*) AS adultos_antes_del_primer_padre
FROM {T}.gold_linea_tiempo l LEFT JOIN pp ON l.player_id = pp.player_id
WHERE l.rol_club = 'adulto' AND (pp.tramo_primer_padre IS NULL OR l.tramo < pp.tramo_primer_padre)"""))

# D: fechas aproximadas y fechas que se corrieron
display(spark.sql(f"""
SELECT SUM(CASE WHEN fecha_desde_aprox    THEN 1 ELSE 0 END) AS desde_aprox,
       SUM(CASE WHEN fecha_desde_ajustada THEN 1 ELSE 0 END) AS desde_ajustada,
       SUM(CASE WHEN fecha_hasta_aprox    THEN 1 ELSE 0 END) AS hasta_aprox,
       SUM(CASE WHEN fecha_hasta_ajustada THEN 1 ELSE 0 END) AS hasta_ajustada
FROM {T}.gold_linea_tiempo"""))

## gold_debut
Una fila por jugador de `silver_players`. El debut es la primera temporada con partidos en un club que no es juvenil según `gold_club_familia`.

| `estado_debut` | Significado |
|---|---|
| `debut_adulto` | Tiene partidos en un club que no es juvenil |
| `solo_juvenil` | Solo tiene partidos en clubes juveniles |
| `sin_apariciones` | No tiene partidos en `silver_player_stats` |

Si en la temporada del debut jugó en varios clubes, se elige el de más apariciones, luego el de más minutos y luego el de menor `club_id`.

Columnas principales: `temporada_debut`, `club_debut_id`, `club_debut_name`, `club_debut_sin_nombre`, `rol_club_debut`, `familia_debut_id`, `apariciones_temporada_debut`, `minutos_temporada_debut`, `n_clubes_temporada_debut`, `club_debut_en_linea` y `familia_debut_en_linea` (si el club o su familia aparecen en la línea de tiempo del jugador), `primera_temporada_juvenil` y `juvenil_antes_del_debut`.

Controles: jugadores por estado, casos para revisar (varios clubes en la temporada de debut, club o familia de debut fuera de su línea de tiempo, juvenil antes del debut) y rango de temporadas con debutantes sin nombre de club.

In [0]:
# EN LIMPIO: gold_debut (escribe workspace.inferiores.gold_debut)
# Debut = primera temporada con partidos en un club que NO es juvenil (según gold_club_familia).
T = "workspace.inferiores"

spark.sql(f"""
WITH st AS (
  SELECT CAST(s.player_id AS STRING) AS player_id,
         CAST(s.season_id AS INT)    AS temporada,
         CAST(s.club_id AS STRING)   AS club_id,
         COALESCE(c.rol, 'adulto')   AS rol_club,
         c.parent_club_id,
         s.appearances, s.minutes_played
  FROM {T}.silver_player_stats s
  LEFT JOIN {T}.gold_club_familia c ON CAST(s.club_id AS STRING) = c.club_id
  WHERE s.appearances > 0
),
adulto AS (
  SELECT player_id, temporada, club_id,
         MAX(parent_club_id) AS parent_club_id, MAX(rol_club) AS rol_club,
         SUM(appearances) AS apariciones, SUM(minutes_played) AS minutos
  FROM st WHERE rol_club <> 'juvenil'
  GROUP BY player_id, temporada, club_id
),
deb AS (
  SELECT player_id, MIN(temporada) AS temporada_debut FROM adulto GROUP BY player_id
),
cand AS (
  SELECT a.*,
         ROW_NUMBER() OVER (PARTITION BY a.player_id
                            ORDER BY a.apariciones DESC, a.minutos DESC NULLS LAST, a.club_id) AS rn,
         COUNT(*) OVER (PARTITION BY a.player_id) AS n_clubes
  FROM adulto a JOIN deb d ON a.player_id = d.player_id AND a.temporada = d.temporada_debut
),
juv AS (
  SELECT player_id, MIN(temporada) AS primera_temporada_juvenil
  FROM st WHERE rol_club = 'juvenil' GROUP BY player_id
),
nom AS (
  SELECT club_id, MAX(nombre) AS club_name
  FROM (
    SELECT CAST(from_club_id AS STRING) AS club_id, from_club_name AS nombre FROM {T}.silver_transfers
    UNION ALL
    SELECT CAST(to_club_id AS STRING), to_club_name FROM {T}.silver_transfers
  ) GROUP BY club_id
),
lt AS (
  SELECT DISTINCT player_id, club_id FROM {T}.gold_linea_tiempo
),
fam AS (
  SELECT DISTINCT player_id, club_familia_id FROM {T}.gold_linea_tiempo
)
SELECT p.player_id,
       CASE WHEN c.player_id IS NOT NULL THEN 'debut_adulto'
            WHEN j.player_id IS NOT NULL THEN 'solo_juvenil'
            ELSE 'sin_apariciones' END                       AS estado_debut,
       c.temporada                                           AS temporada_debut,
       c.club_id                                             AS club_debut_id,
       COALESCE(f.club_name, n.club_name)                    AS club_debut_name,
       CASE WHEN c.club_id IS NULL THEN NULL
            ELSE (COALESCE(f.club_name, n.club_name) IS NULL) END AS club_debut_sin_nombre,
       c.rol_club                                            AS rol_club_debut,
       COALESCE(c.parent_club_id, c.club_id)                 AS familia_debut_id,
       c.apariciones                                         AS apariciones_temporada_debut,
       c.minutos                                             AS minutos_temporada_debut,
       c.n_clubes                                            AS n_clubes_temporada_debut,
       CASE WHEN c.club_id IS NULL THEN NULL
            ELSE (l.club_id IS NOT NULL) END                 AS club_debut_en_linea,
       CASE WHEN c.club_id IS NULL THEN NULL
            ELSE (fm.club_familia_id IS NOT NULL) END        AS familia_debut_en_linea,
       j.primera_temporada_juvenil,
       CASE WHEN c.temporada IS NULL OR j.primera_temporada_juvenil IS NULL THEN NULL
            ELSE (j.primera_temporada_juvenil < c.temporada) END AS juvenil_antes_del_debut
FROM (SELECT DISTINCT CAST(player_id AS STRING) AS player_id FROM {T}.silver_players) p
LEFT JOIN cand c ON c.player_id = p.player_id AND c.rn = 1
LEFT JOIN juv  j ON j.player_id = p.player_id
LEFT JOIN {T}.gold_club_familia f ON f.club_id = c.club_id
LEFT JOIN nom n ON n.club_id = c.club_id
LEFT JOIN lt  l ON l.player_id = p.player_id AND l.club_id = c.club_id
LEFT JOIN fam fm ON fm.player_id = p.player_id
                AND fm.club_familia_id = COALESCE(c.parent_club_id, c.club_id)
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_debut")

# A: filas y estado
display(spark.sql(f"""
SELECT estado_debut, COUNT(*) AS jugadores
FROM {T}.gold_debut GROUP BY estado_debut ORDER BY jugadores DESC"""))
display(spark.sql(f"""
SELECT COUNT(*) AS filas, COUNT(DISTINCT player_id) AS jugadores FROM {T}.gold_debut"""))

# B: casos a mirar
display(spark.sql(f"""
SELECT SUM(CASE WHEN n_clubes_temporada_debut > 1 THEN 1 ELSE 0 END) AS con_varios_clubes,
       SUM(CASE WHEN club_debut_en_linea = false THEN 1 ELSE 0 END)    AS club_debut_fuera_de_su_linea,
       SUM(CASE WHEN familia_debut_en_linea = false THEN 1 ELSE 0 END) AS familia_debut_fuera_de_su_linea,
       SUM(CASE WHEN juvenil_antes_del_debut THEN 1 ELSE 0 END)        AS juvenil_antes_del_debut
FROM {T}.gold_debut"""))

# C: rango de temporadas y debutantes sin nombre de club (clubes que no tienen nombre en ninguna tabla)
display(spark.sql("""
SELECT MIN(temporada_debut) AS primera, MAX(temporada_debut) AS ultima,
       SUM(CASE WHEN club_debut_sin_nombre THEN 1 ELSE 0 END) AS debut_sin_nombre
FROM workspace.inferiores.gold_debut"""))

## gold_formacion_fifa
Reparte la formación del jugador entre los clubes como el mecanismo de solidaridad de la FIFA, sobre un pozo de 100 %: los años calendario de los 12 a los 15 años valen 5 % cada uno y los de los 16 a los 23 años, 10 % cada uno. Las cesiones cuentan para el club que recibe al jugador.

Una fila por jugador, año calendario y club familia.

- **Año de nacimiento:** el de `date_of_birth`. Si falta, se aproxima con el año de `fecha_hasta` menos `edad_hasta` del primer tramo con fecha real (`nacimiento_aprox`).
- **`dias`:** días del año calendario en que el jugador estuvo en esa familia, según `gold_linea_tiempo` y sin los tramos `no_club`. Si un tramo no tiene `fecha_desde`, se cuenta desde el 1 de enero del año en que cumple 12.
- **`peso`:** `peso_edad` (0,05 o 0,10) por `dias` sobre `dias_del_anio`.
- **`peso_relleno`** y **`dias_relleno`:** la parte que sale del tramo inicial, es decir, el relleno hacia atrás hasta el primer club que figura.
- **`hasta_anio_del_debut`:** verdadero si el año es igual o anterior a la temporada de debut.

Columnas: `player_id`, `anio`, `edad_fifa`, `club_familia_id`, `club_familia_name`, `dias`, `dias_del_anio`, `peso_edad`, `peso`, `peso_relleno`, `dias_relleno`, `nacimiento_aprox` y `hasta_anio_del_debut`.

Controles: filas y jugadores con reparto, peso total por jugador (no debe superar 1), rol del club donde empieza el relleno, un jugador al azar año por año y peso en marcadores `no_club` (debe ser 0).

In [0]:
# EN LIMPIO: gold_formacion_fifa (escribe workspace.inferiores.gold_formacion_fifa)
# Reparto de la formación como el mecanismo de solidaridad de la FIFA, sobre un pozo de 100 %:
#   años calendario de los 12 a los 15 años: 5 % c/u; de los 16 a los 23 años: 10 % c/u.
# Una fila por jugador, año calendario y club familia. Las cesiones cuentan para el club que RECIBE al jugador.
T = "workspace.inferiores"

spark.sql(f"""
WITH nac AS (
  SELECT CAST(player_id AS STRING) AS player_id,
         YEAR(CAST(date_of_birth AS DATE)) AS anio_nac, false AS aprox
  FROM {T}.silver_players
  WHERE date_of_birth IS NOT NULL
),
apx AS (
  SELECT player_id, YEAR(fecha_hasta) - edad_hasta AS anio_nac, true AS aprox
  FROM (
    SELECT player_id, fecha_hasta, edad_hasta,
           ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY tramo) AS rn
    FROM {T}.gold_linea_tiempo
    WHERE fecha_hasta IS NOT NULL AND edad_hasta IS NOT NULL AND NOT fecha_hasta_aprox
  ) WHERE rn = 1
),
nacim AS (
  SELECT player_id, anio_nac, aprox FROM nac
  UNION ALL
  SELECT a.player_id, a.anio_nac, a.aprox
  FROM apx a LEFT JOIN nac n ON n.player_id = a.player_id
  WHERE n.player_id IS NULL
),
anios AS (
  SELECT player_id, anio_nac, aprox,
         explode(sequence(anio_nac + 12, anio_nac + 23)) AS anio
  FROM nacim
),
tr AS (
  SELECT player_id, tramo, club_familia_id AS familia_id, tramo_inicial, fecha_desde, fecha_hasta
  FROM {T}.gold_linea_tiempo
  WHERE rol_club <> 'no_club'
),
cruce AS (
  SELECT a.player_id, a.anio, a.anio - a.anio_nac AS edad_fifa, a.aprox AS nacimiento_aprox,
         t.familia_id, t.tramo_inicial,
         GREATEST(0, DATEDIFF(
           LEAST(COALESCE(t.fecha_hasta, current_date()), make_date(a.anio + 1, 1, 1)),
           GREATEST(COALESCE(t.fecha_desde, make_date(a.anio_nac + 12, 1, 1)), make_date(a.anio, 1, 1))
         )) AS dias
  FROM anios a JOIN tr t ON t.player_id = a.player_id
),
agg AS (
  SELECT player_id, anio, edad_fifa, nacimiento_aprox, familia_id,
         SUM(dias) AS dias,
         SUM(CASE WHEN tramo_inicial THEN dias ELSE 0 END) AS dias_relleno
  FROM cruce
  WHERE dias > 0
  GROUP BY player_id, anio, edad_fifa, nacimiento_aprox, familia_id
),
nom AS (
  SELECT club_id, MAX(club_name) AS club_name FROM {T}.gold_linea_tiempo GROUP BY club_id
)
SELECT g.player_id, g.anio, g.edad_fifa, g.familia_id AS club_familia_id,
       COALESCE(f.club_name, n.club_name) AS club_familia_name,
       g.dias,
       DATEDIFF(make_date(g.anio + 1, 1, 1), make_date(g.anio, 1, 1)) AS dias_del_anio,
       CASE WHEN g.edad_fifa <= 15 THEN 0.05 ELSE 0.10 END AS peso_edad,
       (CASE WHEN g.edad_fifa <= 15 THEN 0.05 ELSE 0.10 END)
         * g.dias / DATEDIFF(make_date(g.anio + 1, 1, 1), make_date(g.anio, 1, 1)) AS peso,
       (CASE WHEN g.edad_fifa <= 15 THEN 0.05 ELSE 0.10 END)
         * g.dias_relleno / DATEDIFF(make_date(g.anio + 1, 1, 1), make_date(g.anio, 1, 1)) AS peso_relleno,
       g.dias_relleno,
       g.nacimiento_aprox,
       CASE WHEN d.temporada_debut IS NULL THEN NULL ELSE (g.anio <= d.temporada_debut) END AS hasta_anio_del_debut
FROM agg g
LEFT JOIN {T}.gold_club_familia f ON f.club_id = g.familia_id
LEFT JOIN nom n ON n.club_id = g.familia_id
LEFT JOIN {T}.gold_debut d ON d.player_id = g.player_id
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_formacion_fifa")

# A: tamaño y jugadores con reparto (los que quedaron afuera no tienen año de nacimiento)
display(spark.sql(f"""
SELECT COUNT(*) AS filas,
       COUNT(DISTINCT player_id) AS jugadores_con_reparto,
       (SELECT COUNT(DISTINCT player_id) FROM {T}.gold_linea_tiempo) AS jugadores_en_linea,
       SUM(CASE WHEN nacimiento_aprox THEN 1 ELSE 0 END) AS filas_con_nacimiento_aprox
FROM {T}.gold_formacion_fifa"""))

# B: peso total por jugador (máximo esperado: 1; mayor a 1,0001 es un error de superposición)
display(spark.sql(f"""
WITH t AS (SELECT player_id, SUM(peso) AS total, SUM(peso_relleno) AS relleno
           FROM {T}.gold_formacion_fifa GROUP BY player_id)
SELECT COUNT(*) AS jugadores,
       SUM(CASE WHEN total > 1.0001 THEN 1 ELSE 0 END) AS total_mayor_a_1,
       percentile_approx(total, array(0.1, 0.5, 0.9)) AS total_p10_p50_p90,
       SUM(relleno) / SUM(total) AS proporcion_del_peso_que_es_relleno
FROM t"""))

# C: rol del club donde empieza el relleno (primer tramo de cada jugador)
display(spark.sql(f"""
SELECT rol_club AS rol_del_primer_club, COUNT(*) AS jugadores
FROM {T}.gold_linea_tiempo
WHERE tramo_inicial
GROUP BY rol_club ORDER BY jugadores DESC"""))

# D: un jugador al azar con su reparto, año por año
display(spark.sql(f"""
WITH uno AS (
  SELECT player_id FROM {T}.gold_formacion_fifa
  GROUP BY player_id HAVING COUNT(*) >= 8 ORDER BY md5(player_id) LIMIT 1
)
SELECT g.player_id, g.anio, g.edad_fifa, g.club_familia_name, g.dias, g.peso, g.dias_relleno, g.hasta_anio_del_debut
FROM {T}.gold_formacion_fifa g JOIN uno ON g.player_id = uno.player_id
ORDER BY g.anio, g.dias DESC"""))

# E: peso en marcadores no_club (Unknown, Without Club, etc.) (esperado: 0)
display(spark.sql(f"""
SELECT ROUND(COALESCE(SUM(peso), 0), 4) AS peso_en_marcadores_no_club
FROM {T}.gold_formacion_fifa
WHERE club_familia_id IN (SELECT club_id FROM {T}.bronze_clubes_externos WHERE rol = 'no_club')"""))

## gold_formacion_debut
El mismo reparto que `gold_formacion_fifa`, cortado en el año del debut en un club que no es juvenil y normalizado para que cada jugador sume 1. Los jugadores que no debutaron en un club adulto quedan sin corte. Se descartan los jugadores cuyo peso total queda en 0.

Columnas nuevas: `peso_fifa` y `peso_relleno_fifa` (los de `gold_formacion_fifa`), `peso_norm` y `peso_relleno_norm` (divididos por `peso_total_jugador`), `peso_total_jugador`, `estado_debut` y `temporada_debut`.

Controles: jugadores y filas por estado de debut, que cada jugador sume 1, proporción de relleno con y sin corte, cuánto de la ventana de la FIFA queda después del corte, un jugador al azar y los 10 clubes argentinos con más peso.

In [0]:
# EN LIMPIO: gold_formacion_debut (escribe workspace.inferiores.gold_formacion_debut)
# Mismo reparto que gold_formacion_fifa (5 % por año de los 12 a 15 años, 10 % de los 16 a 23),
# pero cortado en el año del debut en un club que no es juvenil, y normalizado: cada jugador suma 1.
# Los que no debutaron en un club adulto quedan sin corte.
T = "workspace.inferiores"

spark.sql(f"""
WITH base AS (
  SELECT g.player_id, g.anio, g.edad_fifa, g.club_familia_id, g.club_familia_name,
         g.dias, g.dias_del_anio, g.peso_edad, g.peso, g.peso_relleno, g.dias_relleno,
         g.nacimiento_aprox, d.estado_debut, d.temporada_debut
  FROM {T}.gold_formacion_fifa g
  JOIN {T}.gold_debut d ON d.player_id = g.player_id
  WHERE d.estado_debut <> 'debut_adulto' OR g.anio <= d.temporada_debut
),
tot AS (
  SELECT *, SUM(peso) OVER (PARTITION BY player_id) AS peso_total_jugador
  FROM base
)
SELECT player_id, anio, edad_fifa, club_familia_id, club_familia_name,
       dias, dias_del_anio, peso_edad,
       peso              AS peso_fifa,
       peso_relleno      AS peso_relleno_fifa,
       dias_relleno,
       peso / peso_total_jugador         AS peso_norm,
       peso_relleno / peso_total_jugador AS peso_relleno_norm,
       peso_total_jugador,
       estado_debut, temporada_debut, nacimiento_aprox
FROM tot
WHERE peso_total_jugador > 0
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_formacion_debut")

# A: filas y jugadores por estado de debut, y jugadores que perdieron todas sus filas con el corte
display(spark.sql(f"""
SELECT estado_debut, COUNT(DISTINCT player_id) AS jugadores, COUNT(*) AS filas
FROM {T}.gold_formacion_debut GROUP BY estado_debut ORDER BY jugadores DESC"""))
display(spark.sql(f"""
SELECT (SELECT COUNT(DISTINCT player_id) FROM {T}.gold_formacion_fifa)   AS jugadores_en_fifa,
       (SELECT COUNT(DISTINCT player_id) FROM {T}.gold_formacion_debut)  AS jugadores_en_debut"""))

# B: cada jugador suma 1 (esperado: no_suman_1 = 0)
display(spark.sql(f"""
WITH t AS (SELECT player_id, SUM(peso_norm) AS s FROM {T}.gold_formacion_debut GROUP BY player_id)
SELECT COUNT(*) AS jugadores, SUM(CASE WHEN ABS(s - 1) > 0.0001 THEN 1 ELSE 0 END) AS no_suman_1
FROM t"""))

# C: proporción de relleno, con la tabla de la FIFA completa y con el corte en el debut (ambas normalizadas por jugador)
display(spark.sql(f"""
WITH f AS (
  SELECT peso, peso_relleno, SUM(peso) OVER (PARTITION BY player_id) AS pt
  FROM {T}.gold_formacion_fifa
)
SELECT (SELECT SUM(peso_relleno / pt) / SUM(peso / pt) FROM f) AS relleno_fifa_completa,
       (SELECT SUM(peso_relleno_norm) / SUM(peso_norm) FROM {T}.gold_formacion_debut) AS relleno_cortada_en_debut"""))

# D: cuánto de la ventana de la FIFA queda después del corte (percentiles de peso_total_jugador)
display(spark.sql(f"""
WITH t AS (SELECT DISTINCT player_id, peso_total_jugador FROM {T}.gold_formacion_debut)
SELECT percentile_approx(peso_total_jugador, array(0.1, 0.5, 0.9)) AS p10_p50_p90_del_peso_que_queda
FROM t"""))

# E: un jugador al azar que debutó, año por año, con el peso FIFA y el normalizado
display(spark.sql(f"""
WITH uno AS (
  SELECT player_id FROM {T}.gold_formacion_debut
  WHERE estado_debut = 'debut_adulto'
  GROUP BY player_id HAVING COUNT(*) >= 6 ORDER BY md5(player_id) LIMIT 1
)
SELECT g.player_id, g.anio, g.edad_fifa, g.club_familia_name, g.dias,
       g.peso_fifa, g.peso_norm, g.dias_relleno, g.temporada_debut
FROM {T}.gold_formacion_debut g JOIN uno ON g.player_id = uno.player_id
ORDER BY g.anio, g.dias DESC"""))

# F: los 10 clubes argentinos con más peso, con la FIFA completa y con el corte en el debut
display(spark.sql(f"""
WITH a AS (SELECT club_id, club_name FROM {T}.gold_club_familia WHERE origen = 'argentino_105' AND rol = 'padre'),
f AS (SELECT club_familia_id, SUM(peso) AS peso_fifa_completa
      FROM {T}.gold_formacion_fifa GROUP BY club_familia_id),
c AS (SELECT club_familia_id, SUM(peso_norm) AS peso_cortado_en_debut
      FROM {T}.gold_formacion_debut GROUP BY club_familia_id)
SELECT a.club_name,
       ROUND(f.peso_fifa_completa, 1)     AS peso_fifa_completa,
       ROUND(c.peso_cortado_en_debut, 1)  AS peso_cortado_en_debut
FROM a LEFT JOIN f ON f.club_familia_id = a.club_id
       LEFT JOIN c ON c.club_familia_id = a.club_id
ORDER BY c.peso_cortado_en_debut DESC
LIMIT 10"""))

## gold_ranking_formados
Ranking de "formados en" por club familia, con clubes argentinos y del exterior. Parte de `gold_formacion_debut`: cada jugador suma 1, repartido entre sus clubes y cortado en el debut.

| Columnas | Contenido |
|---|---|
| `pais`, `es_argentino`, `en_los_105` | País del club (el más frecuente en `silver_transfers`) y marcas de grupo |
| `jugadores`, `formados_con_relleno`, `formados_sin_relleno` | Todos los jugadores |
| `jugadores_arg`, `formados_arg_con_relleno`, `formados_arg_sin_relleno` | Solo jugadores con ciudadanía argentina |
| `proporcion_relleno_arg` | Parte de `formados_arg_con_relleno` que es relleno |
| `puesto_general`, `puesto_en_su_grupo` | Puesto por `formados_arg_con_relleno`, general y dentro de `es_argentino` |

Controles: suma de todos los jugadores y de los argentinos (un jugador cuenta 1 en total), peso por grupo de club, top 20 de clubes del exterior y top 15 de los clubes de `en_los_105`.

In [0]:
# EN LIMPIO: gold_ranking_formados (escribe workspace.inferiores.gold_ranking_formados)
# Ranking de "formados en" por club familia, con clubes argentinos y del exterior.
#   pais / es_argentino: país del club (el más frecuente en silver_transfers); en_los_105: uno de los 105 padres de la lista.
#   formados_*      : todos los jugadores (cada uno suma 1 repartido entre sus clubes, cortado en el debut).
#   formados_arg_*  : solo jugadores con ciudadanía argentina (silver_player_citizenship).
#   *_con_relleno / *_sin_relleno: con o sin los años rellenados hacia atrás hasta el primer club que figura.
T = "workspace.inferiores"

spark.sql(f"""
WITH pais AS (
  SELECT club_id, pais FROM (
    SELECT club_id, pais,
           ROW_NUMBER() OVER (PARTITION BY club_id ORDER BY n DESC, pais) AS rn
    FROM (
      SELECT club_id, pais, COUNT(*) AS n
      FROM (
        SELECT CAST(from_club_id AS STRING) AS club_id, from_country_name AS pais FROM {T}.silver_transfers
        UNION ALL
        SELECT CAST(to_club_id AS STRING),   to_country_name FROM {T}.silver_transfers
      ) WHERE pais IS NOT NULL
      GROUP BY club_id, pais
    )
  ) WHERE rn = 1
),
arg AS (
  SELECT DISTINCT CAST(player_id AS STRING) AS player_id
  FROM {T}.silver_player_citizenship
  WHERE country = 'Argentina'
),
f AS (
  SELECT g.club_familia_id,
         MAX(g.club_familia_name)                                   AS club_familia_name,
         COUNT(DISTINCT g.player_id)                                AS jugadores,
         COUNT(DISTINCT CASE WHEN a.player_id IS NOT NULL THEN g.player_id END) AS jugadores_arg,
         SUM(g.peso_norm)                                           AS formados_con_relleno,
         SUM(g.peso_norm - g.peso_relleno_norm)                     AS formados_sin_relleno,
         SUM(CASE WHEN a.player_id IS NOT NULL THEN g.peso_norm ELSE 0 END)                         AS formados_arg_con_relleno,
         SUM(CASE WHEN a.player_id IS NOT NULL THEN g.peso_norm - g.peso_relleno_norm ELSE 0 END)   AS formados_arg_sin_relleno
  FROM {T}.gold_formacion_debut g
  LEFT JOIN arg a ON a.player_id = g.player_id
  GROUP BY g.club_familia_id
),
x AS (
  SELECT f.*,
         p.pais                                AS pais,
         COALESCE(p.pais = 'Argentina', false) AS es_argentino,
         (c.club_id IS NOT NULL)               AS en_los_105
  FROM f
  LEFT JOIN pais p ON p.club_id = f.club_familia_id
  LEFT JOIN (SELECT club_id FROM {T}.gold_club_familia
             WHERE origen = 'argentino_105' AND rol = 'padre') c
         ON c.club_id = f.club_familia_id
)
SELECT club_familia_id, club_familia_name, pais, es_argentino, en_los_105,
       jugadores, jugadores_arg,
       ROUND(formados_arg_con_relleno, 4) AS formados_arg_con_relleno,
       ROUND(formados_arg_sin_relleno, 4) AS formados_arg_sin_relleno,
       ROUND(formados_con_relleno, 4)     AS formados_con_relleno,
       ROUND(formados_sin_relleno, 4)     AS formados_sin_relleno,
       ROUND(1 - formados_arg_sin_relleno / NULLIF(formados_arg_con_relleno, 0), 4) AS proporcion_relleno_arg,
       RANK() OVER (ORDER BY formados_arg_con_relleno DESC)                           AS puesto_general,
       RANK() OVER (PARTITION BY es_argentino ORDER BY formados_arg_con_relleno DESC) AS puesto_en_su_grupo
FROM x
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_ranking_formados")

# A: control de sumas (un jugador cuenta 1 en total)
display(spark.sql(f"""
SELECT COUNT(*) AS clubes,
       ROUND(SUM(formados_con_relleno), 2)     AS suma_todos,
       ROUND(SUM(formados_arg_con_relleno), 2) AS suma_solo_argentinos
FROM {T}.gold_ranking_formados"""))

# B: peso por grupo de club (solo jugadores argentinos)
display(spark.sql(f"""
SELECT es_argentino, en_los_105, COUNT(*) AS clubes,
       ROUND(SUM(formados_arg_con_relleno), 2) AS formados_arg_con_relleno,
       ROUND(SUM(formados_arg_sin_relleno), 2) AS formados_arg_sin_relleno
FROM {T}.gold_ranking_formados
GROUP BY es_argentino, en_los_105
ORDER BY es_argentino DESC, en_los_105 DESC"""))

# C: top 20 de clubes del exterior (jugadores argentinos)
display(spark.sql(f"""
SELECT puesto_en_su_grupo, club_familia_name, pais, jugadores_arg,
       formados_arg_con_relleno, formados_arg_sin_relleno, proporcion_relleno_arg
FROM {T}.gold_ranking_formados
WHERE NOT es_argentino
ORDER BY puesto_en_su_grupo LIMIT 20"""))

# D: top 15 de clubes argentinos de los 105 (jugadores argentinos)
display(spark.sql(f"""
SELECT puesto_en_su_grupo, club_familia_name, jugadores_arg,
       formados_arg_con_relleno, formados_arg_sin_relleno, proporcion_relleno_arg
FROM {T}.gold_ranking_formados
WHERE en_los_105
ORDER BY puesto_en_su_grupo LIMIT 15"""))

## gold_salidas
Un evento por cada vez que un jugador pasa de un club juvenil a algo que no es juvenil. Pasar de un juvenil a otro juvenil no cuenta. La cesión sigue siendo del club dueño.

| `categoria` | Cuándo (en este orden) |
|---|---|
| `sin_club` | El destino es un marcador `no_club` |
| `cesion_desde_juvenil` | La familia dueña de destino es la misma y es una cesión |
| `ascenso_primer_equipo` | La familia dueña de destino es la misma y no es una cesión |
| `salida_a_otro_club` | Cualquier otro caso |

Columnas principales: `player_id`, `tramo_origen`, `club_origen_id`, `club_origen_name`, `familia_origen_id`, `club_destino_id`, `club_destino_name`, `familia_destino_id`, `rol_destino`, `categoria`, `fecha_salida`, `fecha_salida_aprox`, `edad_salida`, `n_evento_jugador` y `jugador_arg`.

Controles: eventos y jugadores por categoría, y cantidad de fechas aproximadas, eventos sin fecha y sin edad.

In [0]:
# EN LIMPIO: gold_salidas (escribe workspace.inferiores.gold_salidas)
# Un evento por cada vez que un jugador pasa de un club juvenil a algo que no es juvenil.
# Juvenil -> juvenil no cuenta. La cesión sigue siendo del club dueño.
T = "workspace.inferiores"

spark.sql(f"""
WITH arg AS (
  SELECT DISTINCT CAST(player_id AS STRING) AS player_id
  FROM {T}.silver_player_citizenship WHERE country = 'Argentina'
),
s AS (
  SELECT player_id, tramo, club_id, club_name, rol_club, familia_dueno_id,
         LEAD(club_id)           OVER (PARTITION BY player_id ORDER BY tramo) AS sig_club_id,
         LEAD(club_name)         OVER (PARTITION BY player_id ORDER BY tramo) AS sig_club_name,
         LEAD(rol_club)          OVER (PARTITION BY player_id ORDER BY tramo) AS sig_rol,
         LEAD(familia_dueno_id)  OVER (PARTITION BY player_id ORDER BY tramo) AS sig_dueno,
         LEAD(esta_cedido)       OVER (PARTITION BY player_id ORDER BY tramo) AS sig_cedido,
         LEAD(fecha_desde)       OVER (PARTITION BY player_id ORDER BY tramo) AS fecha_salida,
         LEAD(fecha_desde_aprox) OVER (PARTITION BY player_id ORDER BY tramo) AS fecha_salida_aprox,
         LEAD(edad_desde)        OVER (PARTITION BY player_id ORDER BY tramo) AS edad_salida
  FROM {T}.gold_linea_tiempo
)
SELECT s.player_id,
       s.tramo                AS tramo_origen,
       s.club_id              AS club_origen_id,
       s.club_name            AS club_origen_name,
       s.familia_dueno_id     AS familia_origen_id,
       s.sig_club_id          AS club_destino_id,
       s.sig_club_name        AS club_destino_name,
       s.sig_dueno            AS familia_destino_id,
       s.sig_rol              AS rol_destino,
       CASE WHEN s.sig_rol = 'no_club'                                  THEN 'sin_club'
            WHEN s.sig_dueno = s.familia_dueno_id AND s.sig_cedido      THEN 'cesion_desde_juvenil'
            WHEN s.sig_dueno = s.familia_dueno_id                       THEN 'ascenso_primer_equipo'
            ELSE 'salida_a_otro_club' END AS categoria,
       s.fecha_salida, s.fecha_salida_aprox, s.edad_salida,
       ROW_NUMBER() OVER (PARTITION BY s.player_id ORDER BY s.tramo) AS n_evento_jugador,
       (a.player_id IS NOT NULL) AS jugador_arg
FROM s LEFT JOIN arg a ON a.player_id = s.player_id
WHERE s.rol_club = 'juvenil' AND s.sig_rol IS NOT NULL AND s.sig_rol <> 'juvenil'
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_salidas")

# A: eventos por categoría
display(spark.sql(f"""
SELECT categoria, COUNT(*) AS eventos, COUNT(DISTINCT player_id) AS jugadores
FROM {T}.gold_salidas GROUP BY categoria ORDER BY eventos DESC"""))
display(spark.sql(f"SELECT COUNT(*) AS eventos_total, COUNT(DISTINCT player_id) AS jugadores FROM {T}.gold_salidas"))

# B: fechas aproximadas y edad vacía en la salida
display(spark.sql(f"""
SELECT SUM(CASE WHEN fecha_salida_aprox THEN 1 ELSE 0 END) AS fecha_aprox,
       SUM(CASE WHEN fecha_salida IS NULL THEN 1 ELSE 0 END) AS sin_fecha,
       SUM(CASE WHEN edad_salida IS NULL THEN 1 ELSE 0 END)  AS sin_edad
FROM {T}.gold_salidas"""))

## gold_ranking_salidos
"Salidos de" por club familia: jugadores distintos por categoría de `gold_salidas`, para todos y solo con ciudadanía argentina (columnas `_arg`).

Columnas: `club_familia_id`, `club_familia_name`, `pais`, `es_argentino`, `en_los_105`, `eventos_total`, `jugadores_total`, `salidas_a_otro_club`, `ascensos`, `cesiones` y `sin_club` (cada una con su versión `_arg`), `puesto_general` y `puesto_en_su_grupo`. Los puestos se calculan por `salidas_a_otro_club_arg`.

Controles: suma de eventos y clubes sin nombre, top 15 de los clubes de `en_los_105` y top 10 de clubes del exterior.

In [0]:
# EN LIMPIO: gold_ranking_salidos (escribe workspace.inferiores.gold_ranking_salidos)
# "Salidos de" por club familia: jugadores distintos por categoría, para todos y solo con ciudadanía argentina (_arg).
T = "workspace.inferiores"

spark.sql(f"""
WITH pais AS (
  SELECT club_id, pais FROM (
    SELECT club_id, pais, ROW_NUMBER() OVER (PARTITION BY club_id ORDER BY n DESC, pais) AS rn
    FROM (
      SELECT club_id, pais, COUNT(*) AS n
      FROM (
        SELECT CAST(from_club_id AS STRING) AS club_id, from_country_name AS pais FROM {T}.silver_transfers
        UNION ALL
        SELECT CAST(to_club_id AS STRING), to_country_name FROM {T}.silver_transfers
      ) WHERE pais IS NOT NULL
      GROUP BY club_id, pais
    )
  ) WHERE rn = 1
),
nom AS (
  SELECT club_id, MAX(club_name) AS club_name FROM {T}.gold_linea_tiempo GROUP BY club_id
),
agg AS (
  SELECT familia_origen_id AS club_familia_id,
         COUNT(*)                         AS eventos_total,
         COUNT(DISTINCT player_id)        AS jugadores_total,
         COUNT(DISTINCT CASE WHEN categoria = 'salida_a_otro_club'     THEN player_id END)                  AS salidas_a_otro_club,
         COUNT(DISTINCT CASE WHEN categoria = 'salida_a_otro_club'     AND jugador_arg THEN player_id END)  AS salidas_a_otro_club_arg,
         COUNT(DISTINCT CASE WHEN categoria = 'ascenso_primer_equipo'  THEN player_id END)                  AS ascensos,
         COUNT(DISTINCT CASE WHEN categoria = 'ascenso_primer_equipo'  AND jugador_arg THEN player_id END)  AS ascensos_arg,
         COUNT(DISTINCT CASE WHEN categoria = 'cesion_desde_juvenil'   THEN player_id END)                  AS cesiones,
         COUNT(DISTINCT CASE WHEN categoria = 'cesion_desde_juvenil'   AND jugador_arg THEN player_id END)  AS cesiones_arg,
         COUNT(DISTINCT CASE WHEN categoria = 'sin_club'               THEN player_id END)                  AS sin_club,
         COUNT(DISTINCT CASE WHEN categoria = 'sin_club'               AND jugador_arg THEN player_id END)  AS sin_club_arg
  FROM {T}.gold_salidas
  GROUP BY familia_origen_id
),
x AS (
  SELECT agg.*, COALESCE(f.club_name, n.club_name) AS club_familia_name,
         p.pais AS pais,
         COALESCE(p.pais = 'Argentina', false) AS es_argentino,
         (g.club_id IS NOT NULL) AS en_los_105
  FROM agg
  LEFT JOIN {T}.gold_club_familia f ON f.club_id = agg.club_familia_id
  LEFT JOIN nom n ON n.club_id = agg.club_familia_id
  LEFT JOIN pais p ON p.club_id = agg.club_familia_id
  LEFT JOIN (SELECT club_id FROM {T}.gold_club_familia WHERE origen = 'argentino_105' AND rol = 'padre') g
         ON g.club_id = agg.club_familia_id
)
SELECT *,
       RANK() OVER (ORDER BY salidas_a_otro_club_arg DESC)                           AS puesto_general,
       RANK() OVER (PARTITION BY es_argentino ORDER BY salidas_a_otro_club_arg DESC) AS puesto_en_su_grupo
FROM x
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_ranking_salidos")

# A: control (la suma de eventos_total debe dar el total de eventos de gold_salidas y clubes_sin_nombre debe dar 0)
display(spark.sql(f"""
SELECT COUNT(*) AS clubes, SUM(eventos_total) AS eventos,
       SUM(CASE WHEN club_familia_name IS NULL THEN 1 ELSE 0 END) AS clubes_sin_nombre
FROM {T}.gold_ranking_salidos"""))

# B: top 15 de los 105 por salidas a otro club (jugadores argentinos)
display(spark.sql(f"""
SELECT puesto_en_su_grupo, club_familia_name, salidas_a_otro_club_arg, ascensos_arg, cesiones_arg, sin_club_arg
FROM {T}.gold_ranking_salidos
WHERE en_los_105 ORDER BY salidas_a_otro_club_arg DESC LIMIT 15"""))

# C: top 10 de clubes del exterior
display(spark.sql(f"""
SELECT puesto_en_su_grupo, club_familia_name, pais, salidas_a_otro_club_arg, ascensos_arg, cesiones_arg
FROM {T}.gold_ranking_salidos
WHERE NOT es_argentino ORDER BY salidas_a_otro_club_arg DESC LIMIT 10"""))

## gold_ranking_camada
Una fila por club familia y camada, solo con jugadores de ciudadanía argentina. La camada es el año de debut en un club que no es juvenil; una camada vacía agrupa a los jugadores sin debut en un club adulto.

Combina `gold_formacion_debut` (formados) con `gold_salidas` (salidas) y toma los datos del club de `gold_ranking_formados` y `gold_ranking_salidos`.

Columnas: `club_familia_id`, `club_familia_name`, `pais`, `es_argentino`, `en_los_105`, `camada`, `jugadores_arg`, `formados_arg_con_relleno`, `formados_arg_sin_relleno`, `salidas_a_otro_club_arg`, `ascensos_arg`, `cesiones_arg`, `sin_club_arg` y `puesto_en_camada` (puesto dentro de la camada por `formados_arg_con_relleno`).

Controles: que las sumas coincidan con los rankings por club, rango de camadas y filas sin camada, los 15 mejores pares club-camada de los clubes de `en_los_105` y la evolución del club con más formados desde el año 2000.

In [0]:
# EN LIMPIO: gold_ranking_camada (escribe workspace.inferiores.gold_ranking_camada)
# Camada = año de debut del jugador en un club que no es juvenil (temporada_debut de gold_debut).
# Una fila por club familia y camada. camada vacía = jugadores sin debut en un club adulto.
# Solo jugadores con ciudadanía argentina (_arg).
T = "workspace.inferiores"

spark.sql(f"""
WITH arg AS (
  SELECT DISTINCT CAST(player_id AS STRING) AS player_id
  FROM {T}.silver_player_citizenship WHERE country = 'Argentina'
),
dbt AS (
  SELECT player_id, temporada_debut AS camada FROM {T}.gold_debut
),
form AS (
  SELECT g.club_familia_id, d.camada,
         COUNT(DISTINCT g.player_id)                         AS jugadores_arg,
         SUM(g.peso_norm)                                    AS formados_arg_con_relleno,
         SUM(g.peso_norm - g.peso_relleno_norm)              AS formados_arg_sin_relleno
  FROM {T}.gold_formacion_debut g
  JOIN arg a ON a.player_id = g.player_id
  JOIN dbt d ON d.player_id = g.player_id
  GROUP BY g.club_familia_id, d.camada
),
sal AS (
  SELECT s.familia_origen_id AS club_familia_id, d.camada,
         COUNT(DISTINCT CASE WHEN s.categoria = 'salida_a_otro_club'    THEN s.player_id END) AS salidas_a_otro_club_arg,
         COUNT(DISTINCT CASE WHEN s.categoria = 'ascenso_primer_equipo' THEN s.player_id END) AS ascensos_arg,
         COUNT(DISTINCT CASE WHEN s.categoria = 'cesion_desde_juvenil'  THEN s.player_id END) AS cesiones_arg,
         COUNT(DISTINCT CASE WHEN s.categoria = 'sin_club'              THEN s.player_id END) AS sin_club_arg
  FROM {T}.gold_salidas s
  JOIN arg a ON a.player_id = s.player_id
  JOIN dbt d ON d.player_id = s.player_id
  GROUP BY s.familia_origen_id, d.camada
),
j AS (
  SELECT COALESCE(f.club_familia_id, s.club_familia_id) AS club_familia_id,
         COALESCE(f.camada, s.camada) AS camada,
         COALESCE(f.jugadores_arg, 0)             AS jugadores_arg,
         COALESCE(f.formados_arg_con_relleno, 0)  AS formados_arg_con_relleno,
         COALESCE(f.formados_arg_sin_relleno, 0)  AS formados_arg_sin_relleno,
         COALESCE(s.salidas_a_otro_club_arg, 0)   AS salidas_a_otro_club_arg,
         COALESCE(s.ascensos_arg, 0)              AS ascensos_arg,
         COALESCE(s.cesiones_arg, 0)              AS cesiones_arg,
         COALESCE(s.sin_club_arg, 0)              AS sin_club_arg
  FROM form f
  FULL OUTER JOIN sal s ON f.club_familia_id = s.club_familia_id AND f.camada <=> s.camada
)
SELECT j.club_familia_id,
       COALESCE(r.club_familia_name, q.club_familia_name) AS club_familia_name,
       COALESCE(r.pais, q.pais)                           AS pais,
       COALESCE(r.es_argentino, q.es_argentino, false)    AS es_argentino,
       COALESCE(r.en_los_105, q.en_los_105, false)        AS en_los_105,
       j.camada,
       j.jugadores_arg,
       ROUND(j.formados_arg_con_relleno, 4) AS formados_arg_con_relleno,
       ROUND(j.formados_arg_sin_relleno, 4) AS formados_arg_sin_relleno,
       j.salidas_a_otro_club_arg, j.ascensos_arg, j.cesiones_arg, j.sin_club_arg,
       RANK() OVER (PARTITION BY j.camada ORDER BY j.formados_arg_con_relleno DESC) AS puesto_en_camada
FROM j
LEFT JOIN {T}.gold_ranking_formados r ON r.club_familia_id = j.club_familia_id
LEFT JOIN {T}.gold_ranking_salidos  q ON q.club_familia_id = j.club_familia_id
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_ranking_camada")

# A: control contra los rankings por club (los tres pares deben coincidir)
display(spark.sql(f"""
SELECT (SELECT ROUND(SUM(formados_arg_con_relleno), 2) FROM {T}.gold_ranking_camada)  AS formados_camada,
       (SELECT ROUND(SUM(formados_arg_con_relleno), 2) FROM {T}.gold_ranking_formados) AS formados_club,
       (SELECT SUM(salidas_a_otro_club_arg) FROM {T}.gold_ranking_camada)  AS salidas_camada,
       (SELECT SUM(salidas_a_otro_club_arg) FROM {T}.gold_ranking_salidos) AS salidas_club"""))

# B: rango de camadas y filas sin camada
display(spark.sql(f"""
SELECT MIN(camada) AS primera_camada, MAX(camada) AS ultima_camada,
       SUM(CASE WHEN camada IS NULL THEN 1 ELSE 0 END) AS filas_sin_debut,
       COUNT(*) AS filas, SUM(CASE WHEN club_familia_name IS NULL THEN 1 ELSE 0 END) AS filas_sin_nombre
FROM {T}.gold_ranking_camada"""))

# C: los 15 mejores pares club-camada entre los 105 (por formados con relleno)
display(spark.sql(f"""
SELECT club_familia_name, camada, jugadores_arg, formados_arg_con_relleno, formados_arg_sin_relleno,
       salidas_a_otro_club_arg, ascensos_arg
FROM {T}.gold_ranking_camada
WHERE en_los_105 AND camada IS NOT NULL
ORDER BY formados_arg_con_relleno DESC LIMIT 15"""))

# D: el club de los 105 con más formados, año de debut por año de debut desde el 2000
display(spark.sql(f"""
WITH top AS (
  SELECT club_familia_id FROM {T}.gold_ranking_formados
  WHERE en_los_105 ORDER BY formados_arg_con_relleno DESC LIMIT 1
)
SELECT c.club_familia_name, c.camada, c.jugadores_arg, c.formados_arg_con_relleno,
       c.salidas_a_otro_club_arg, c.ascensos_arg, c.puesto_en_camada
FROM {T}.gold_ranking_camada c JOIN top ON top.club_familia_id = c.club_familia_id
WHERE c.camada >= 2000 ORDER BY c.camada"""))

## gold_jugador
Una fila por jugador de `silver_players`. Es la dimensión de jugadores para Power BI.

- `es_argentino` y `ciudadanias`: ciudadanía argentina y lista de ciudadanías en el orden de la respuesta.
- `anio_nacimiento` y `nacimiento_aprox`: el año de `date_of_birth` o, si falta, el año del primer traspaso con fecha real menos la edad (la misma regla que `gold_formacion_fifa`).
- `club_familia_id`: la familia del club actual.

Columnas: nombres, `url`, fechas de nacimiento y fallecimiento, país y ciudad de nacimiento, altura, pie, posición (`position_main`, `position_group`), retiro, datos del club actual y de su contrato, `club_familia_id`, `es_argentino` y `ciudadanias`.

Controles: filas contra `silver_players`, nulos por columna, jugadores por grupo de posición, ciudadanía y nacimiento, club actual fuera de `gold_club_familia` y una muestra al azar.

In [0]:
# gold_jugador: una fila por jugador (dimensión para Power BI). EN LIMPIO.
# Lee: silver_players, silver_player_citizenship, gold_club_familia, gold_linea_tiempo.
# es_argentino = ciudadanía argentina (misma regla que los rankings).
# nacimiento_aprox: para jugadores sin date_of_birth, año de nacimiento = año del primer
# traspaso con fecha real menos la age (igual que gold_formacion_fifa).
T = "workspace.inferiores"

spark.sql(f"""
WITH cit AS (
  SELECT CAST(player_id AS STRING) AS player_id,
         array_join(transform(array_sort(collect_list(struct(orden, country))), x -> x.country), ', ') AS ciudadanias,
         MAX(CASE WHEN country = 'Argentina' THEN 1 ELSE 0 END) = 1 AS es_argentino
  FROM {T}.silver_player_citizenship
  GROUP BY CAST(player_id AS STRING)
),
apx AS (
  SELECT player_id, YEAR(fecha_hasta) - edad_hasta AS anio_nac_aprox
  FROM (
    SELECT player_id, fecha_hasta, edad_hasta,
           ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY tramo) AS rn
    FROM {T}.gold_linea_tiempo
    WHERE fecha_hasta IS NOT NULL AND edad_hasta IS NOT NULL AND NOT fecha_hasta_aprox
  ) WHERE rn = 1
)
SELECT CAST(p.player_id AS STRING)                         AS player_id,
       p.name, p.name_in_home_country, p.url,
       p.date_of_birth, p.date_of_death,
       COALESCE(YEAR(p.date_of_birth), a.anio_nac_aprox)   AS anio_nacimiento,
       (p.date_of_birth IS NULL AND a.anio_nac_aprox IS NOT NULL) AS nacimiento_aprox,
       p.birth_country, p.birth_city, p.height_cm, p.foot,
       p.position_main, p.position_group,
       p.is_retired, p.retired_since,
       CAST(p.club_id AS STRING)                           AS club_id,
       p.club_name, p.club_joined, p.contract_expires,
       CASE WHEN p.club_id IS NULL THEN NULL
            ELSE COALESCE(f.parent_club_id, CAST(p.club_id AS STRING)) END AS club_familia_id,
       COALESCE(c.es_argentino, false)                     AS es_argentino,
       c.ciudadanias
FROM {T}.silver_players p
LEFT JOIN cit c ON c.player_id = CAST(p.player_id AS STRING)
LEFT JOIN apx a ON a.player_id = CAST(p.player_id AS STRING)
LEFT JOIN {T}.gold_club_familia f ON f.club_id = CAST(p.club_id AS STRING)
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_jugador")

# ---------- Chequeos (generales, sin ids a mano) ----------
g = spark.table(f"{T}.gold_jugador")
print("1) filas / jugadores distintos / jugadores en silver_players")
display(spark.sql(f"""
SELECT (SELECT COUNT(*) FROM {T}.gold_jugador)                         AS filas,
       (SELECT COUNT(DISTINCT player_id) FROM {T}.gold_jugador)        AS jugadores,
       (SELECT COUNT(DISTINCT player_id) FROM {T}.silver_players)      AS en_silver
"""))

print("2) nulos por columna")
display(g.selectExpr(*[f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`" for c in g.columns]))

print("3) por posición (grupo)")
display(spark.sql(f"""
SELECT position_group, COUNT(*) AS jugadores, SUM(CASE WHEN es_argentino THEN 1 ELSE 0 END) AS argentinos
FROM {T}.gold_jugador GROUP BY position_group ORDER BY jugadores DESC
"""))

print("4) ciudadanía y nacimiento")
display(spark.sql(f"""
SELECT SUM(CASE WHEN es_argentino THEN 1 ELSE 0 END)            AS es_argentino,
       SUM(CASE WHEN ciudadanias IS NULL THEN 1 ELSE 0 END)     AS sin_ciudadania,
       SUM(CASE WHEN date_of_birth IS NOT NULL THEN 1 ELSE 0 END) AS con_fecha_nacimiento,
       SUM(CASE WHEN nacimiento_aprox THEN 1 ELSE 0 END)        AS nacimiento_aprox,
       SUM(CASE WHEN anio_nacimiento IS NULL THEN 1 ELSE 0 END) AS sin_ningun_nacimiento
FROM {T}.gold_jugador
"""))

print("5) club actual: cuántos tienen club y cuántos no están en gold_club_familia")
display(spark.sql(f"""
SELECT SUM(CASE WHEN j.club_id IS NOT NULL THEN 1 ELSE 0 END)                         AS con_club_actual,
       SUM(CASE WHEN j.club_id IS NOT NULL AND f.club_id IS NULL THEN 1 ELSE 0 END)    AS club_fuera_de_gold_club_familia
FROM {T}.gold_jugador j LEFT JOIN {T}.gold_club_familia f ON f.club_id = j.club_id
"""))

print("6) muestra al azar")
display(spark.sql(f"""
SELECT player_id, name, anio_nacimiento, nacimiento_aprox, position_main, club_name,
       club_familia_id, es_argentino, ciudadanias
FROM {T}.gold_jugador ORDER BY md5(player_id) LIMIT 10
"""))

## gold_seleccion
Una fila por jugador con al menos una selección, a partir de `silver_national_team_career`.

Las columnas `partidos_*` y `goles_*` se calculan por categoría. Nulo significa que el jugador nunca tuvo esa selección; 0 significa que la tuvo y no jugó.

| Categoría | Regla |
|---|---|
| `mayor`, `olimpica`, `b` | `categoria` igual a ese valor |
| `u23`, `u21`, `u20`, `u19`, `u18`, `u17`, `u16`, `u15` | `categoria = juvenil` y `edad_max` igual a la edad |

Otras columnas: `n_selecciones`, `selecciones` (lista en el orden de la respuesta), `partidos_total`, `goles_total`, `partidos_mayor_arg` y `goles_mayor_arg` (solo la selección mayor llamada `Argentina`), `primer_debut`, `debut_mayor`, `ultimo_partido` y `es_capitan_mayor`.

Controles: jugadores contra silver y contra `gold_jugador`, totales de partidos y goles contra silver, jugadores, partidos y goles por categoría, nulos por columna y una muestra al azar.

In [0]:
# gold_seleccion: una fila por jugador con al menos una selección. EN LIMPIO.
# Columnas partidos_* / goles_* por categoría: null = nunca tuvo esa selección; 0 = la tuvo y no jugó.
# Lee: silver_national_team_career. Las juveniles se separan por edad_max (U15 ... U23).
T = "workspace.inferiores"

cats = [("mayor",   "categoria = 'mayor'"),
        ("olimpica","categoria = 'olimpica'"),
        ("u23",     "categoria = 'juvenil' AND edad_max = 23"),
        ("u21",     "categoria = 'juvenil' AND edad_max = 21"),
        ("u20",     "categoria = 'juvenil' AND edad_max = 20"),
        ("u19",     "categoria = 'juvenil' AND edad_max = 19"),
        ("u18",     "categoria = 'juvenil' AND edad_max = 18"),
        ("u17",     "categoria = 'juvenil' AND edad_max = 17"),
        ("u16",     "categoria = 'juvenil' AND edad_max = 16"),
        ("u15",     "categoria = 'juvenil' AND edad_max = 15"),
        ("b",       "categoria = 'b'")]

cols_cat = ",\n       ".join(
    f"SUM(CASE WHEN {c} THEN appearances END) AS partidos_{n},\n"
    f"       SUM(CASE WHEN {c} THEN goals END)       AS goles_{n}" for n, c in cats)

spark.sql(f"""
SELECT CAST(player_id AS STRING) AS player_id,
       COUNT(*)                                   AS n_selecciones,
       array_join(transform(array_sort(collect_list(struct(orden, team_name))), x -> x.team_name), ', ') AS selecciones,
       SUM(appearances)                           AS partidos_total,
       SUM(goals)                                 AS goles_total,
       {cols_cat},
       SUM(CASE WHEN categoria = 'mayor' AND team_name = 'Argentina' THEN appearances END) AS partidos_mayor_arg,
       SUM(CASE WHEN categoria = 'mayor' AND team_name = 'Argentina' THEN goals END)       AS goles_mayor_arg,
       MIN(debut)                                 AS primer_debut,
       MIN(CASE WHEN categoria = 'mayor' THEN debut END) AS debut_mayor,
       MAX(last_match)                            AS ultimo_partido,
       MAX(CASE WHEN categoria = 'mayor' AND is_captain THEN 1 ELSE 0 END) = 1 AS es_capitan_mayor
FROM {T}.silver_national_team_career
GROUP BY CAST(player_id AS STRING)
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_seleccion")

# ---------- Chequeos (generales, sin ids a mano) ----------
g = spark.table(f"{T}.gold_seleccion")

print("1) filas / jugadores distintos / jugadores en silver / jugadores que no están en gold_jugador")
display(spark.sql(f"""
SELECT (SELECT COUNT(*) FROM {T}.gold_seleccion)                                   AS filas,
       (SELECT COUNT(DISTINCT player_id) FROM {T}.gold_seleccion)                  AS jugadores,
       (SELECT COUNT(DISTINCT player_id) FROM {T}.silver_national_team_career)     AS en_silver,
       (SELECT COUNT(*) FROM {T}.gold_seleccion s
         LEFT JOIN {T}.gold_jugador j ON j.player_id = s.player_id
         WHERE j.player_id IS NULL)                                                AS fuera_de_gold_jugador
"""))

print("2) totales: gold vs silver (tienen que coincidir)")
display(spark.sql(f"""
SELECT (SELECT SUM(partidos_total) FROM {T}.gold_seleccion)          AS partidos_gold,
       (SELECT SUM(appearances) FROM {T}.silver_national_team_career) AS partidos_silver,
       (SELECT SUM(goles_total) FROM {T}.gold_seleccion)             AS goles_gold,
       (SELECT SUM(goals) FROM {T}.silver_national_team_career)      AS goles_silver,
       (SELECT SUM(CASE WHEN appearances IS NULL OR goals IS NULL THEN 1 ELSE 0 END)
          FROM {T}.silver_national_team_career)                      AS filas_silver_con_nulos
"""))

print("3) por categoría: jugadores que la tuvieron, partidos y goles")
filas = []
for n, _ in cats:
    r = g.selectExpr(f"'{n}' AS categoria",
                     f"SUM(CASE WHEN partidos_{n} IS NOT NULL THEN 1 ELSE 0 END) AS jugadores",
                     f"SUM(partidos_{n}) AS partidos", f"SUM(goles_{n}) AS goles").collect()[0]
    filas.append((r["categoria"], r["jugadores"], r["partidos"], r["goles"]))
display(spark.createDataFrame(filas, "categoria string, jugadores long, partidos long, goles long"))

print("4) nulos por columna")
display(g.selectExpr(*[f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`" for c in g.columns]))

print("5) muestra al azar")
display(spark.sql(f"""
SELECT player_id, n_selecciones, selecciones, partidos_total, goles_total,
       partidos_mayor, goles_mayor, partidos_u20, goles_u20, es_capitan_mayor
FROM {T}.gold_seleccion ORDER BY md5(player_id) LIMIT 10
"""))

## gold_valor_mercado_serie y gold_valor_mercado
Solo incluye jugadores de `gold_jugador`.

**`gold_valor_mercado_serie`:** un punto de valor por fila, a partir de `silver_market_value_history`. Un valor nulo es un momento sin valoración, no un 0. Columnas: `player_id`, `orden`, `fecha`, `anio`, `edad`, `valor`, `club_id`, `club_name`, `club_familia_id` y `es_ultimo_punto`.

**`gold_valor_mercado`:** resumen por jugador.

| Columnas | Contenido |
|---|---|
| `n_puntos`, `n_valoraciones`, `primera_valoracion`, `ultima_valoracion` | Puntos de la serie y valoraciones con valor |
| `valor_max` y sus `fecha_`, `edad_`, `club_id_`, `club_name_`, `club_familia_id_` | Valor máximo y el momento en que se dio |
| `valor_actual`, `fecha_valor_actual` | Último punto de la serie |
| `area_valor_meur_anios` | Valor vigente hasta el siguiente punto, en millones de euros por año. Un valor nulo suma 0, el último punto no suma y es nulo si hay un solo punto |
| `fee_max`, `fee_total`, `fee_total_traspasos`, `fee_total_cesiones`, `n_pases_con_fee` | Solo pases con `fee` mayor a 0 y sin `upcoming`. El `fee` es un entero sin moneda |

La tabla une valor y `fee` con un `FULL OUTER JOIN`: un jugador con `fee` y sin historial de valor tiene `n_puntos` nulo.

Controles: serie contra silver, filas y jugadores del resumen, nulos por columna, `fee` contra silver, valor actual, área y club del valor máximo, top 10 por área y una muestra al azar.

In [0]:
# gold_valor_mercado_serie (un punto de valor por fila) y gold_valor_mercado (resumen por jugador). EN LIMPIO.
# Lee: silver_market_value_history, silver_transfers (solo fee), gold_jugador (universo), gold_club_familia.
# Solo jugadores que están en gold_jugador: los de silver_market_value_history que no están en silver_players quedan afuera.
# valor null = momento sin valoración (no es 0). area_valor_meur_anios: valor vigente hasta el
# siguiente punto, en M€·años (null suma 0; el último punto no suma, no tiene "siguiente"; null si hay 1 solo punto).
# fee: solo pases con fee > 0 y sin upcoming; entero sin moneda. Traspasos = 'transfer', cesiones = 'loan'.
T = "workspace.inferiores"

# ---------- 1) Serie ----------
spark.sql(f"""
WITH m AS (
  SELECT CAST(h.player_id AS STRING) AS player_id, h.orden, h.date AS fecha, h.age AS edad,
         h.market_value AS valor, CAST(h.club_id AS STRING) AS club_id, h.club_name,
         ROW_NUMBER() OVER (PARTITION BY h.player_id ORDER BY h.date DESC, h.orden DESC) AS rn_ult
  FROM {T}.silver_market_value_history h
  LEFT SEMI JOIN {T}.gold_jugador j ON j.player_id = CAST(h.player_id AS STRING)
)
SELECT m.player_id, m.orden, m.fecha, YEAR(m.fecha) AS anio, m.edad, m.valor,
       m.club_id, m.club_name,
       COALESCE(f.parent_club_id, m.club_id) AS club_familia_id,
       (m.rn_ult = 1)                        AS es_ultimo_punto
FROM m LEFT JOIN {T}.gold_club_familia f ON f.club_id = m.club_id
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_valor_mercado_serie")

# ---------- 2) Resumen por jugador ----------
spark.sql(f"""
WITH s AS (
  SELECT player_id, orden, fecha, edad, valor, club_id, club_name, club_familia_id,
         LEAD(fecha) OVER (PARTITION BY player_id ORDER BY fecha, orden) AS fecha_siguiente,
         ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY valor DESC NULLS LAST, fecha, orden) AS rn_max,
         ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY fecha DESC, orden DESC)              AS rn_ult
  FROM {T}.gold_valor_mercado_serie
),
v AS (
  SELECT player_id,
         COUNT(*)                                                          AS n_puntos,
         COUNT(valor)                                                      AS n_valoraciones,
         MIN(CASE WHEN valor IS NOT NULL THEN fecha END)                   AS primera_valoracion,
         MAX(CASE WHEN valor IS NOT NULL THEN fecha END)                   AS ultima_valoracion,
         MAX(CASE WHEN rn_max = 1 AND valor IS NOT NULL THEN valor END)    AS valor_max,
         MAX(CASE WHEN rn_max = 1 AND valor IS NOT NULL THEN fecha END)    AS fecha_valor_max,
         MAX(CASE WHEN rn_max = 1 AND valor IS NOT NULL THEN edad END)     AS edad_valor_max,
         MAX(CASE WHEN rn_max = 1 AND valor IS NOT NULL THEN club_id END)  AS club_id_valor_max,
         MAX(CASE WHEN rn_max = 1 AND valor IS NOT NULL THEN club_name END) AS club_name_valor_max,
         MAX(CASE WHEN rn_max = 1 AND valor IS NOT NULL THEN club_familia_id END) AS club_familia_id_valor_max,
         MAX(CASE WHEN rn_ult = 1 THEN valor END)                          AS valor_actual,
         MAX(CASE WHEN rn_ult = 1 THEN fecha END)                          AS fecha_valor_actual,
         ROUND(SUM(CASE WHEN fecha_siguiente IS NOT NULL
                        THEN COALESCE(valor, 0) * datediff(fecha_siguiente, fecha) / 365.25 END) / 1e6, 2)
                                                                           AS area_valor_meur_anios
  FROM s GROUP BY player_id
),
fee AS (
  SELECT CAST(t.player_id AS STRING) AS player_id,
         MAX(CASE WHEN t.fee > 0 THEN t.fee END)                                  AS fee_max,
         SUM(CASE WHEN t.fee > 0 THEN t.fee END)                                  AS fee_total,
         SUM(CASE WHEN t.fee > 0 AND t.transfer_type = 'transfer' THEN t.fee END) AS fee_total_traspasos,
         SUM(CASE WHEN t.fee > 0 AND t.transfer_type = 'loan' THEN t.fee END)     AS fee_total_cesiones,
         SUM(CASE WHEN t.fee > 0 THEN 1 ELSE 0 END)                               AS n_pases_con_fee
  FROM {T}.silver_transfers t
  LEFT SEMI JOIN {T}.gold_jugador j ON j.player_id = CAST(t.player_id AS STRING)
  WHERE NOT COALESCE(t.upcoming, false)
  GROUP BY CAST(t.player_id AS STRING)
  HAVING SUM(CASE WHEN t.fee > 0 THEN 1 ELSE 0 END) > 0
)
SELECT COALESCE(v.player_id, fee.player_id) AS player_id,
       v.n_puntos, v.n_valoraciones, v.primera_valoracion, v.ultima_valoracion,
       v.valor_max, v.fecha_valor_max, v.edad_valor_max,
       v.club_id_valor_max, v.club_name_valor_max, v.club_familia_id_valor_max,
       v.valor_actual, v.fecha_valor_actual, v.area_valor_meur_anios,
       fee.fee_max, fee.fee_total, fee.fee_total_traspasos, fee.fee_total_cesiones, fee.n_pases_con_fee
FROM v FULL OUTER JOIN fee ON fee.player_id = v.player_id
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_valor_mercado")

# ---------- Chequeos (generales, sin ids a mano) ----------
r = spark.table(f"{T}.gold_valor_mercado")

print("1) serie: gold contra silver (dentro y fuera de gold_jugador)")
display(spark.sql(f"""
SELECT (SELECT COUNT(*) FROM {T}.gold_valor_mercado_serie)                                      AS serie_filas,
       (SELECT COUNT(*) FROM {T}.silver_market_value_history h
          LEFT SEMI JOIN {T}.gold_jugador j ON j.player_id = CAST(h.player_id AS STRING))       AS silver_filas_en_gold_jugador,
       (SELECT COUNT(*) FROM {T}.silver_market_value_history)                                   AS silver_filas_total,
       (SELECT COUNT(DISTINCT player_id) FROM {T}.gold_valor_mercado_serie)                     AS serie_jugadores,
       (SELECT SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) FROM {T}.gold_valor_mercado_serie) AS serie_valor_null,
       (SELECT MAX(valor) FROM {T}.gold_valor_mercado_serie)                                    AS serie_max,
       (SELECT SUM(CASE WHEN es_ultimo_punto THEN 1 ELSE 0 END) FROM {T}.gold_valor_mercado_serie) AS ultimos_puntos
"""))

print("2) resumen: filas, jugadores, con historial, solo fee, sin ningún valor, fuera de gold_jugador")
display(spark.sql(f"""
SELECT COUNT(*) AS filas, COUNT(DISTINCT r.player_id) AS jugadores,
       SUM(CASE WHEN r.n_puntos IS NOT NULL THEN 1 ELSE 0 END) AS con_historial,
       SUM(CASE WHEN r.n_puntos IS NULL THEN 1 ELSE 0 END)     AS solo_fee,
       SUM(CASE WHEN r.n_valoraciones = 0 THEN 1 ELSE 0 END)   AS sin_ningun_valor,
       SUM(CASE WHEN j.player_id IS NULL THEN 1 ELSE 0 END)    AS fuera_de_gold_jugador,
       SUM(r.n_puntos) AS suma_puntos
FROM {T}.gold_valor_mercado r LEFT JOIN {T}.gold_jugador j ON j.player_id = r.player_id
"""))

print("3) nulos por columna del resumen")
display(r.selectExpr(*[f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`" for c in r.columns]))

print("4) fee: gold contra silver (dentro de gold_jugador), y las partes")
display(spark.sql(f"""
SELECT (SELECT SUM(fee_total) FROM {T}.gold_valor_mercado)           AS fee_total_gold,
       (SELECT SUM(t.fee) FROM {T}.silver_transfers t
          LEFT SEMI JOIN {T}.gold_jugador j ON j.player_id = CAST(t.player_id AS STRING)
          WHERE NOT COALESCE(t.upcoming, false) AND t.fee > 0)       AS fee_total_silver,
       (SELECT SUM(fee_total_traspasos) FROM {T}.gold_valor_mercado) AS traspasos,
       (SELECT SUM(fee_total_cesiones) FROM {T}.gold_valor_mercado)  AS cesiones,
       (SELECT SUM(n_pases_con_fee) FROM {T}.gold_valor_mercado)     AS pases_gold,
       (SELECT MAX(fee_max) FROM {T}.gold_valor_mercado)             AS fee_max_gold,
       (SELECT COUNT(*) FROM {T}.gold_valor_mercado WHERE fee_max IS NOT NULL) AS jugadores_con_fee
"""))

print("5) valor actual, área y club del valor máximo")
display(spark.sql(f"""
SELECT SUM(CASE WHEN n_puntos IS NOT NULL AND valor_actual IS NULL THEN 1 ELSE 0 END)  AS sin_valoracion_vigente,
       SUM(CASE WHEN valor_actual IS NOT NULL AND fecha_valor_actual >= DATE'2026-01-01' THEN 1 ELSE 0 END) AS con_valor_actual_2026,
       SUM(CASE WHEN area_valor_meur_anios IS NULL AND n_puntos IS NOT NULL THEN 1 ELSE 0 END) AS area_null_con_historial,
       SUM(CASE WHEN club_id_valor_max IS NOT NULL AND club_name_valor_max IS NULL THEN 1 ELSE 0 END) AS club_max_sin_nombre,
       SUM(CASE WHEN club_id_valor_max IS NOT NULL AND f.club_id IS NULL THEN 1 ELSE 0 END)    AS club_max_fuera_de_gold_club_familia
FROM {T}.gold_valor_mercado r LEFT JOIN {T}.gold_club_familia f ON f.club_id = r.club_id_valor_max
"""))

print("6) top 10 por área (sentido común)")
display(spark.sql(f"""
SELECT j.name, r.valor_max, r.edad_valor_max, r.n_valoraciones, r.area_valor_meur_anios, r.fee_max
FROM {T}.gold_valor_mercado r JOIN {T}.gold_jugador j ON j.player_id = r.player_id
ORDER BY r.area_valor_meur_anios DESC NULLS LAST LIMIT 10
"""))

print("7) muestra al azar")
display(spark.sql(f"""
SELECT r.player_id, j.name, r.n_valoraciones, r.valor_max, r.fecha_valor_max, r.valor_actual,
       r.fecha_valor_actual, r.area_valor_meur_anios, r.fee_max, r.fee_total, r.fee_total_traspasos, r.fee_total_cesiones
FROM {T}.gold_valor_mercado r LEFT JOIN {T}.gold_jugador j ON j.player_id = r.player_id
ORDER BY md5(r.player_id) LIMIT 10
"""))       

## gold_minutos
Una fila por jugador, temporada, competición y club, solo con filas con partidos, a partir de `silver_player_stats`. Solo jugadores de `gold_jugador`.

`rol_club` es el de `gold_club_familia` o `adulto` si el club no está ahí (la misma regla que `gold_debut`). No incluye nivel de liga.

Columnas: `player_id`, `temporada`, `competition_id`, `competition_name`, `club_id`, `club_name`, `club_familia_id`, `rol_club`, `partidos`, `minutos`, `goles`, `asistencias`, `amarillas`, `dobles_amarillas` y `rojas`.

Controles: filas y claves únicas, totales contra silver, nulos por columna, filas por rol de club, rango de temporadas y clubes sin nombre, y una muestra al azar.

In [0]:
# gold_minutos: una fila por jugador, temporada, competición y club (solo filas con partidos). EN LIMPIO.
# Lee: silver_player_stats, silver_transfers (solo nombres de club), gold_club_familia, gold_jugador (universo).
# rol_club = COALESCE(rol, 'adulto'), igual que gold_debut: un club ausente de gold_club_familia se trata como adulto.
# Sin nivel de liga: no está decidido.
T = "workspace.inferiores"

spark.sql(f"""
WITH nom AS (
  SELECT club_id, MAX(nombre) AS club_name FROM (
    SELECT CAST(from_club_id AS STRING) AS club_id, from_club_name AS nombre FROM {T}.silver_transfers
    UNION ALL
    SELECT CAST(to_club_id AS STRING), to_club_name FROM {T}.silver_transfers
  ) WHERE club_id IS NOT NULL AND nombre IS NOT NULL GROUP BY club_id
)
SELECT CAST(s.player_id AS STRING)                          AS player_id,
       CAST(s.season_id AS INT)                             AS temporada,
       s.competition_id, s.competition_name,
       CAST(s.club_id AS STRING)                            AS club_id,
       COALESCE(f.club_name, n.club_name)                   AS club_name,
       COALESCE(f.parent_club_id, CAST(s.club_id AS STRING)) AS club_familia_id,
       COALESCE(f.rol, 'adulto')                            AS rol_club,
       s.appearances AS partidos, s.minutes_played AS minutos,
       s.goals AS goles, s.assists AS asistencias,
       s.yellow_cards AS amarillas, s.second_yellow_cards AS dobles_amarillas, s.red_cards AS rojas
FROM {T}.silver_player_stats s
LEFT JOIN {T}.gold_club_familia f ON f.club_id = CAST(s.club_id AS STRING)
LEFT JOIN nom n ON n.club_id = CAST(s.club_id AS STRING)
WHERE s.appearances > 0
  AND CAST(s.player_id AS STRING) IN (SELECT player_id FROM {T}.gold_jugador)
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_minutos")

# ---------- Chequeos (generales, sin ids a mano) ----------
g = spark.table(f"{T}.gold_minutos")

print("1) filas, claves únicas, jugadores y jugadores de gold_jugador sin ninguna fila")
display(spark.sql(f"""
SELECT COUNT(*) AS filas,
       COUNT(DISTINCT concat_ws('|', player_id, temporada, competition_id, club_id)) AS claves_distintas,
       COUNT(DISTINCT player_id) AS jugadores,
       (SELECT COUNT(*) FROM {T}.gold_jugador) - COUNT(DISTINCT player_id) AS jugadores_sin_partidos,
       SUM(CASE WHEN player_id NOT IN (SELECT player_id FROM {T}.gold_jugador) THEN 1 ELSE 0 END) AS fuera_de_gold_jugador
FROM {T}.gold_minutos
"""))

print("2) totales: gold contra silver (solo jugadores de gold_jugador, con partidos)")
display(spark.sql(f"""
SELECT (SELECT SUM(partidos) FROM {T}.gold_minutos)  AS partidos_gold,
       (SELECT SUM(appearances) FROM {T}.silver_player_stats
         WHERE appearances > 0 AND CAST(player_id AS STRING) IN (SELECT player_id FROM {T}.gold_jugador)) AS partidos_silver,
       (SELECT SUM(minutos) FROM {T}.gold_minutos)   AS minutos_gold,
       (SELECT SUM(minutes_played) FROM {T}.silver_player_stats
         WHERE appearances > 0 AND CAST(player_id AS STRING) IN (SELECT player_id FROM {T}.gold_jugador)) AS minutos_silver,
       (SELECT SUM(goles) FROM {T}.gold_minutos)     AS goles_gold,
       (SELECT SUM(goals) FROM {T}.silver_player_stats
         WHERE appearances > 0 AND CAST(player_id AS STRING) IN (SELECT player_id FROM {T}.gold_jugador)) AS goles_silver
"""))

print("3) nulos por columna")
display(g.selectExpr(*[f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`" for c in g.columns]))

print("4) por rol del club")
display(spark.sql(f"""
SELECT rol_club, COUNT(*) AS filas, COUNT(DISTINCT player_id) AS jugadores,
       SUM(partidos) AS partidos, SUM(minutos) AS minutos
FROM {T}.gold_minutos GROUP BY rol_club ORDER BY filas DESC
"""))

print("5) rango de temporadas y clubes sin nombre")
display(spark.sql(f"""
SELECT MIN(temporada) AS desde, MAX(temporada) AS hasta, COUNT(DISTINCT competition_id) AS competiciones,
       COUNT(DISTINCT club_id) AS clubes,
       SUM(CASE WHEN club_name IS NULL THEN 1 ELSE 0 END) AS filas_club_sin_nombre,
       COUNT(DISTINCT CASE WHEN club_name IS NULL THEN club_id END) AS clubes_sin_nombre
FROM {T}.gold_minutos
"""))

print("6) muestra al azar")
display(spark.sql(f"""
SELECT m.player_id, j.name, m.temporada, m.competition_name, m.club_name, m.rol_club,
       m.partidos, m.minutos, m.goles, m.asistencias
FROM {T}.gold_minutos m LEFT JOIN {T}.gold_jugador j ON j.player_id = m.player_id
ORDER BY md5(concat_ws('|', m.player_id, m.temporada, m.competition_id, m.club_id)) LIMIT 10
"""))

## gold_logros
Una fila por detalle de cada logro (jugador, título, temporada, club), a partir de `silver_achievements`. Solo jugadores de `gold_jugador`.

`tipo_logro` se clasifica con reglas sobre el texto del título, en inglés, español, alemán, italiano y portugués. El orden importa: `premio_individual` va antes que `titulo`, porque "Winner Ballon d'Or" es un premio.

| `tipo_logro` | El título (en minúsculas) |
|---|---|
| `premio_individual` | Menciona goleador, jugador del año o de la temporada, mejor jugador o jugador joven, Balón de Oro, bota, balón o guante de oro, o MVP |
| `participacion` | Contiene `participant` o `participante` |
| `tercer_puesto` | Contiene `third place` o `3rd place` |
| `subcampeon` | Contiene `runner`, `subcampe`, `finalist` o `second place` |
| `ascenso` | Empieza con `promotion` |
| `descenso` | Empieza con `relegated` |
| `medalla` | Contiene `medalist` |
| `titulo` | Contiene `winner`, `champion`, `campe`, `campione`, `meister`, `sieger` o `scudetto` |
| `otro` | Ninguna de las anteriores |

`es_seleccion` es verdadero cuando el club figura como equipo en `silver_national_team_career`; en esas filas `club_familia_id` es nulo.

Controles: filas y claves únicas contra silver, logros por tipo, lo que quedó en `otro`, títulos sin competición o con palabras de premio, selecciones detectadas, nulos por columna y una muestra al azar.

In [0]:
# gold_logros: una fila por detalle de cada logro (jugador, título, temporada, club). EN LIMPIO.
# Lee: silver_achievements, silver_national_team_career (solo para marcar selecciones), gold_club_familia, gold_jugador.
# tipo_logro se clasifica por reglas sobre el texto del título (en inglés, español, alemán, italiano, portugués).
# El orden de las reglas importa: premio_individual va antes que titulo ("Winner Ballon d'Or" es un premio).
# es_seleccion: el club figura como equipo en silver_national_team_career; para esas filas club_familia_id = null.
T = "workspace.inferiores"

tipo = """CASE
  WHEN lower(a.title) RLIKE 'top goal ?scorer|top scorer|footballer of the year|player of the (year|season)|best (fifa|player|young)|ballon|golden (boot|ball|glove|boy)|mvp|young player' THEN 'premio_individual'
  WHEN lower(a.title) RLIKE 'participant|participante' THEN 'participacion'
  WHEN lower(a.title) RLIKE 'third place|3rd place' THEN 'tercer_puesto'
  WHEN lower(a.title) RLIKE 'runner|subcampe|finalist|second place' THEN 'subcampeon'
  WHEN lower(a.title) LIKE 'promotion%' THEN 'ascenso'
  WHEN lower(a.title) LIKE 'relegated%' THEN 'descenso'
  WHEN lower(a.title) LIKE '%medalist%' THEN 'medalla'
  WHEN lower(a.title) RLIKE 'winner|champion|campe|campione|meister|sieger|scudetto' THEN 'titulo'
  ELSE 'otro' END"""

spark.sql(f"""
WITH sel AS (SELECT DISTINCT CAST(team_id AS STRING) AS team_id FROM {T}.silver_national_team_career)
SELECT CAST(a.player_id AS STRING)            AS player_id,
       a.orden_titulo, a.orden_detalle,
       a.title                                AS titulo_original,
       {tipo}                                 AS tipo_logro,
       CAST(a.season_id AS INT)               AS temporada,
       a.season_name,
       a.competition_id, a.competition_name,
       CAST(a.club_id AS STRING)              AS club_id,
       a.club_name,
       (sel.team_id IS NOT NULL)              AS es_seleccion,
       CASE WHEN a.club_id IS NULL OR sel.team_id IS NOT NULL THEN NULL
            ELSE COALESCE(f.parent_club_id, CAST(a.club_id AS STRING)) END AS club_familia_id
FROM {T}.silver_achievements a
LEFT JOIN sel ON sel.team_id = CAST(a.club_id AS STRING)
LEFT JOIN {T}.gold_club_familia f ON f.club_id = CAST(a.club_id AS STRING)
WHERE CAST(a.player_id AS STRING) IN (SELECT player_id FROM {T}.gold_jugador)
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{T}.gold_logros")

# ---------- Chequeos (generales, sin ids a mano) ----------
g = spark.table(f"{T}.gold_logros")

print("1) filas, claves únicas, jugadores, y contra silver (jugadores de gold_jugador)")
display(spark.sql(f"""
SELECT COUNT(*) AS filas,
       COUNT(DISTINCT concat_ws('|', player_id, orden_titulo, orden_detalle)) AS claves_distintas,
       COUNT(DISTINCT player_id) AS jugadores,
       (SELECT COUNT(*) FROM {T}.silver_achievements
         WHERE CAST(player_id AS STRING) IN (SELECT player_id FROM {T}.gold_jugador)) AS filas_silver,
       (SELECT COUNT(*) FROM {T}.silver_achievements) AS filas_silver_total,
       SUM(CASE WHEN player_id NOT IN (SELECT player_id FROM {T}.gold_jugador) THEN 1 ELSE 0 END) AS fuera_de_gold_jugador
FROM {T}.gold_logros
"""))

print("2) por tipo de logro")
display(spark.sql(f"""
SELECT tipo_logro, COUNT(DISTINCT titulo_original) AS titulos_distintos, COUNT(*) AS filas,
       COUNT(DISTINCT player_id) AS jugadores
FROM {T}.gold_logros GROUP BY tipo_logro ORDER BY filas DESC
"""))

print("3) lo que quedó en 'otro'")
display(spark.sql(f"""
SELECT titulo_original, COUNT(*) AS filas FROM {T}.gold_logros
WHERE tipo_logro = 'otro' GROUP BY titulo_original ORDER BY filas DESC
"""))

print("4) control: 'titulo' sin competición, o con palabras de premio (¿hay premios mal clasificados?)")
display(spark.sql(f"""
SELECT titulo_original, COUNT(*) AS filas,
       SUM(CASE WHEN competition_id IS NULL THEN 1 ELSE 0 END) AS sin_competicion
FROM {T}.gold_logros
WHERE tipo_logro = 'titulo'
  AND (competition_id IS NULL OR lower(titulo_original) RLIKE 'best|player|award|golden|boot|glove|top |mvp|revelation')
GROUP BY titulo_original ORDER BY filas DESC LIMIT 40
"""))

print("5) selecciones: filas, por club, y 'Argentina...' no detectadas")
display(spark.sql(f"""
SELECT SUM(CASE WHEN es_seleccion THEN 1 ELSE 0 END) AS filas_de_seleccion,
       SUM(CASE WHEN NOT es_seleccion AND club_name LIKE 'Argentina%' THEN 1 ELSE 0 END) AS argentina_no_detectada,
       SUM(CASE WHEN club_familia_id IS NULL AND club_id IS NOT NULL AND NOT es_seleccion THEN 1 ELSE 0 END) AS club_sin_familia
FROM {T}.gold_logros
"""))
display(spark.sql(f"""
SELECT club_name, tipo_logro, COUNT(*) AS filas FROM {T}.gold_logros
WHERE es_seleccion GROUP BY club_name, tipo_logro ORDER BY filas DESC LIMIT 12
"""))

print("6) nulos por columna")
display(g.selectExpr(*[f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`" for c in g.columns]))

print("7) muestra al azar")
display(spark.sql(f"""
SELECT l.player_id, j.name, l.tipo_logro, l.titulo_original, l.temporada, l.competition_name, l.club_name, l.es_seleccion
FROM {T}.gold_logros l LEFT JOIN {T}.gold_jugador j ON j.player_id = l.player_id
ORDER BY md5(concat_ws('|', l.player_id, l.orden_titulo, l.orden_detalle)) LIMIT 12
"""))

## gold_competicion
Una fila por `competition_id` de `gold_minutos` o `gold_logros`. La categoría sale de `bronze_competiciones`; el país y el carácter internacional se calculan con el país de los clubes que juegan la competición.

| Parámetro | Valor | Para qué sirve |
|---|---|---|
| `UMBRAL_INTERNACIONAL` | `0.80` | Proporción de clubes de un mismo país a partir de la cual la competición es de ese país |
| `MIN_CLUBES_PAIS` | `3` | Mínimo de clubes con país para decidir el ámbito |

**Categorías de Transfermarkt a tipo** (`map_categoria_tm`)

| Categoría | `tipo` |
|---|---|
| `First Tier` a `Tenth Tier` | `liga`, con `nivel` de 1 a 10 |
| `Youth league` | `liga_juvenil` (juvenil) |
| `Reserve league` | `liga_reserva` (juvenil) |
| `Domestic Cup` | `copa_nacional` |
| `League Cup` | `copa_liga` |
| `Further Cup` | `copa_otra` |
| `Domestic Super Cup`, `Further Super Cup` | `supercopa` |
| `Play-Offs` | `playoffs` |
| `Regional Championship` | `campeonato_regional` |
| `Domestic Youth Cup`, `National Youth Super Cup`, `International Youth Cup` | `copa_juvenil` (juvenil) |

**`ambito`**, en este orden: `selecciones` si la mitad o más de sus "clubes" son selecciones; `internacional` si la categoría es `International Youth Cup`; `nacional` si Transfermarkt le puso categoría; `sin_dato` si hay menos de `MIN_CLUBES_PAIS` clubes con país; `internacional` si ningún país junta `UMBRAL_INTERNACIONAL` de los clubes; y `nacional` en el resto.

**`tipo_competicion`:** `seleccion` para las selecciones; el `tipo` de la categoría (o `otra_categoria_tm`) si tiene categoría; `internacional_clubes`; `nacional_sin_categoria`; o `sin_clasificar`.

`pais` solo se completa para las nacionales en las que un país junta `UMBRAL_INTERNACIONAL` o más de los clubes. `competicion_grupo` es "país - categoría" en esas, y el nombre de la competición en las demás. `nivel` y `es_juvenil` quedan nulos para las selecciones.

La celda usa tres vistas temporales: `map_categoria_tm`, `pais_club` y `comp_stats`.

Controles: filas e IDs únicos, competiciones de minutos o logros sin fila, tipo y ámbito, nacionales sin país, los 15 grupos más grandes y nulos por columna.

In [0]:
# gold_competicion: una fila por competition_id de gold_minutos o gold_logros. EN LIMPIO.
# Lee: bronze_competiciones (categoría de Transfermarkt), gold_minutos, gold_logros, silver_transfers (país de cada club).
# Sin listas por competición: la categoría viene de Transfermarkt (vocabulario cerrado, traducido una vez abajo) y el país
# y el carácter internacional se calculan con el país de los clubes que la juegan.
# ambito (en este orden): selecciones (la mitad o más de sus "clubes" son selecciones) > internacional (International Youth Cup)
#   > nacional (si Transfermarkt le puso categoría) > sin_dato (menos de MIN_CLUBES_PAIS clubes con país)
#   > internacional (ningún país junta UMBRAL_INTERNACIONAL de los clubes) > nacional.
# pais y competicion_grupo solo se completan si un país junta UMBRAL_INTERNACIONAL o más de los clubes.
T = "workspace.inferiores"
UMBRAL_INTERNACIONAL = 0.80
MIN_CLUBES_PAIS = 3

_tiers = ["First", "Second", "Third", "Fourth", "Fifth", "Sixth", "Seventh", "Eighth", "Ninth", "Tenth"]
CAT = [(f"{n} Tier", "liga", i + 1, False) for i, n in enumerate(_tiers)] + [
    ("Youth league", "liga_juvenil", None, True),
    ("Reserve league", "liga_reserva", None, True),
    ("Domestic Cup", "copa_nacional", None, False),
    ("League Cup", "copa_liga", None, False),
    ("Further Cup", "copa_otra", None, False),
    ("Domestic Super Cup", "supercopa", None, False),
    ("Further Super Cup", "supercopa", None, False),
    ("Play-Offs", "playoffs", None, False),
    ("Regional Championship", "campeonato_regional", None, False),
    ("Domestic Youth Cup", "copa_juvenil", None, True),
    ("National Youth Super Cup", "copa_juvenil", None, True),
    ("International Youth Cup", "copa_juvenil", None, True)]
spark.createDataFrame(CAT, "categoria_tm string, tipo string, nivel int, juvenil boolean") \
     .createOrReplaceTempView("map_categoria_tm")

# País de cada club (el más frecuente en silver_transfers) y clubes de cada competición
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW pais_club AS
WITH x AS (
  SELECT CAST(from_club_id AS STRING) AS club_id, from_country_name AS pais FROM {T}.silver_transfers
  WHERE from_club_id IS NOT NULL AND from_country_name IS NOT NULL
  UNION ALL
  SELECT CAST(to_club_id AS STRING), to_country_name FROM {T}.silver_transfers
  WHERE to_club_id IS NOT NULL AND to_country_name IS NOT NULL),
y AS (SELECT club_id, pais, COUNT(*) AS n FROM x GROUP BY club_id, pais)
SELECT club_id, pais, ROW_NUMBER() OVER (PARTITION BY club_id ORDER BY n DESC, pais) AS rk FROM y
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW comp_stats AS
WITH comp_clubes AS (
  SELECT DISTINCT competition_id, club_id, false AS es_sel FROM {T}.gold_minutos
  UNION
  SELECT DISTINCT competition_id, club_id, es_seleccion FROM {T}.gold_logros
  WHERE competition_id IS NOT NULL AND club_id IS NOT NULL),
cc AS (SELECT c.competition_id, c.club_id, c.es_sel, p.pais
       FROM comp_clubes c LEFT JOIN pais_club p ON p.club_id = c.club_id AND p.rk = 1),
cp AS (SELECT competition_id, pais, COUNT(*) AS n FROM cc WHERE NOT es_sel AND pais IS NOT NULL GROUP BY competition_id, pais),
tp AS (SELECT competition_id, pais AS pais_top, n,
              ROW_NUMBER() OVER (PARTITION BY competition_id ORDER BY n DESC, pais) AS r FROM cp),
t AS (SELECT competition_id, COUNT(*) AS n_clubes,
             SUM(CASE WHEN es_sel THEN 1 ELSE 0 END) AS n_selecciones,
             SUM(CASE WHEN NOT es_sel AND pais IS NOT NULL THEN 1 ELSE 0 END) AS n_clubes_con_pais,
             COUNT(DISTINCT CASE WHEN NOT es_sel THEN pais END) AS n_paises
      FROM cc GROUP BY competition_id)
SELECT t.*, tp.pais_top, ROUND(tp.n / NULLIF(t.n_clubes_con_pais, 0), 3) AS share_pais_top,
       ROUND(t.n_selecciones / t.n_clubes, 3) AS share_selecciones
FROM t LEFT JOIN tp ON tp.competition_id = t.competition_id AND tp.r = 1
""")

spark.sql(f"""
WITH b AS (
  SELECT competition_id, nombre AS nombre_tm, categoria_tm, fuente_categoria, estado AS estado_tm,
         ROW_NUMBER() OVER (PARTITION BY competition_id ORDER BY visitado_en DESC) AS rn
  FROM {T}.bronze_competiciones),
base AS (
  SELECT competition_id, MAX(competition_name) AS competition_name,
         SUM(filas_minutos) AS filas_minutos, SUM(filas_logros) AS filas_logros
  FROM (SELECT competition_id, competition_name, COUNT(*) AS filas_minutos, 0 AS filas_logros
        FROM {T}.gold_minutos GROUP BY competition_id, competition_name
        UNION ALL
        SELECT competition_id, competition_name, 0, COUNT(*)
        FROM {T}.gold_logros WHERE competition_id IS NOT NULL GROUP BY competition_id, competition_name)
  GROUP BY competition_id),
z AS (
  SELECT base.competition_id, base.competition_name, base.filas_minutos, base.filas_logros,
         b.nombre_tm, b.categoria_tm, b.fuente_categoria, b.estado_tm,
         m.tipo AS tipo_cat, m.nivel AS nivel_cat, m.juvenil AS juvenil_cat,
         s.n_clubes, s.n_clubes_con_pais, s.n_paises, s.n_selecciones, s.share_selecciones, s.pais_top, s.share_pais_top,
         CASE WHEN s.share_selecciones >= 0.5 THEN 'selecciones'
              WHEN b.categoria_tm = 'International Youth Cup' THEN 'internacional'
              WHEN b.categoria_tm IS NOT NULL THEN 'nacional'
              WHEN COALESCE(s.n_clubes_con_pais, 0) < {MIN_CLUBES_PAIS} THEN 'sin_dato'
              WHEN s.share_pais_top < {UMBRAL_INTERNACIONAL} THEN 'internacional'
              ELSE 'nacional' END AS ambito
  FROM base
  LEFT JOIN (SELECT * FROM b WHERE rn = 1) b ON b.competition_id = base.competition_id
  LEFT JOIN map_categoria_tm m ON m.categoria_tm = b.categoria_tm
  LEFT JOIN comp_stats s ON s.competition_id = base.competition_id)
SELECT competition_id, competition_name, nombre_tm, categoria_tm, fuente_categoria, estado_tm,
       CASE WHEN ambito = 'selecciones' THEN 'seleccion'
            WHEN categoria_tm IS NOT NULL THEN COALESCE(tipo_cat, 'otra_categoria_tm')
            WHEN ambito = 'internacional' THEN 'internacional_clubes'
            WHEN ambito = 'nacional' THEN 'nacional_sin_categoria'
            ELSE 'sin_clasificar' END                                          AS tipo_competicion,
       ambito,
       CASE WHEN ambito <> 'selecciones' THEN nivel_cat END                     AS nivel,
       CASE WHEN ambito = 'nacional' AND share_pais_top >= {UMBRAL_INTERNACIONAL} THEN pais_top END AS pais,
       CASE WHEN ambito <> 'selecciones' THEN juvenil_cat END                   AS es_juvenil,
       CASE WHEN ambito = 'nacional' AND pais_top IS NOT NULL AND categoria_tm IS NOT NULL
                 AND share_pais_top >= {UMBRAL_INTERNACIONAL}
            THEN concat(pais_top, ' - ', categoria_tm) ELSE competition_name END AS competicion_grupo,
       n_clubes, n_clubes_con_pais, n_paises, share_pais_top, n_selecciones, share_selecciones,
       filas_minutos, filas_logros
FROM z
""").write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{T}.gold_competicion")

# ---------- Chequeos (generales, sin ids a mano) ----------
g = spark.table(f"{T}.gold_competicion")

print("1) filas, ids únicos, y competiciones de minutos o logros sin fila (esperado: filas = ids, 0, 0)")
display(spark.sql(f"""
SELECT COUNT(*) AS filas, COUNT(DISTINCT competition_id) AS ids,
       (SELECT COUNT(DISTINCT competition_id) FROM {T}.gold_minutos
         WHERE competition_id NOT IN (SELECT competition_id FROM {T}.gold_competicion)) AS minutos_sin_fila,
       (SELECT COUNT(DISTINCT competition_id) FROM {T}.gold_logros
         WHERE competition_id IS NOT NULL AND competition_id NOT IN (SELECT competition_id FROM {T}.gold_competicion)) AS logros_sin_fila
FROM {T}.gold_competicion
"""))

print("2) por tipo y ámbito")
display(spark.sql(f"""
SELECT tipo_competicion, ambito, COUNT(*) AS competiciones,
       SUM(filas_minutos) AS filas_minutos, SUM(filas_logros) AS filas_logros
FROM {T}.gold_competicion GROUP BY tipo_competicion, ambito ORDER BY filas_minutos DESC
"""))

print("3) nacionales sin país y las que tienen país con 80% o menos de los clubes (esperado: 0)")
display(spark.sql(f"""
SELECT SUM(CASE WHEN pais IS NULL THEN 1 ELSE 0 END) AS nacionales_sin_pais,
       SUM(CASE WHEN pais IS NOT NULL AND share_pais_top < {UMBRAL_INTERNACIONAL} THEN 1 ELSE 0 END) AS pais_con_poca_evidencia
FROM {T}.gold_competicion WHERE ambito = 'nacional'
"""))
display(spark.sql(f"""
SELECT competition_id, competition_name, categoria_tm, pais, competicion_grupo, n_clubes_con_pais, n_paises, share_pais_top
FROM {T}.gold_competicion WHERE ambito = 'nacional' AND n_clubes_con_pais >= {MIN_CLUBES_PAIS} AND share_pais_top < {UMBRAL_INTERNACIONAL}
"""))

print("4) competicion_grupo: los 15 grupos más grandes y cuántos ids juntan")
display(spark.sql(f"""
SELECT competicion_grupo, ambito, tipo_competicion, COUNT(*) AS ids, SUM(filas_minutos) AS filas_minutos
FROM {T}.gold_competicion GROUP BY competicion_grupo, ambito, tipo_competicion ORDER BY filas_minutos DESC LIMIT 15
"""))

print("5) nulos por columna")
display(g.selectExpr(*[f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`" for c in g.columns]))

## gold_aporte
Una fila por jugador y club formador, con los pesos de formación, métricas del jugador, datos del club y la camada. La celda crea la tabla con `CREATE OR REPLACE TABLE`.

| Columnas | Contenido |
|---|---|
| `fifa`, `fifa_sin_relleno` | Peso del jugador en el club según `gold_formacion_fifa` |
| `debut`, `debut_sin_relleno` | Peso según `gold_formacion_debut` |
| `minutos_no_juv`, `goles_no_juv`, `asistencias_no_juv` | Minutos, goles y asistencias en clubes que no son juveniles |
| `partidos_7_grandes` | Partidos en las competiciones `GB1`, `ES1`, `IT1`, `L1`, `FR1`, `PO1` y `NL1` |
| `partidos_champions` | Partidos en la competición `CL` |
| `partidos_mayor`, `goles_mayor` | Solo con la selección mayor de Argentina (`gold_seleccion`) |
| `titulos` | Cantidad de logros de tipo `titulo` |
| `vallas_invictas` | Suma de `clean_sheets` de `silver_player_stats` |
| `position_group`, `position_main` | Posición del jugador (`Sin posición` si falta) |
| `club_familia_name`, `pais`, `es_argentino`, `en_los_105` | Datos del club (`Sin nombre (<id>)` si no tiene nombre en ninguna tabla) |
| `camada`, `lustro_desde`, `lustro` | Año de debut y lustro (por ejemplo `2000-2004`, o `Sin debut`) para segmentar los rankings por época |

Controles: filas, jugadores y clubes con sumas de pesos, blancos en los datos del club y jugadores por lustro.

In [0]:
%python
# gold_aporte: una fila por jugador y club formador (pesos de los 4 criterios + métricas del jugador
# + datos del club: nombre, país, es_argentino, en_los_105; salen para TODOS los clubes de la tabla)
# + camada y lustro (año de debut en un club adulto, de gold_debut) para segmentar rankings por época
T = "workspace.inferiores"
GRANDES = "'GB1','ES1','IT1','L1','FR1','PO1','NL1'"

spark.sql(f"""
CREATE OR REPLACE TABLE {T}.gold_aporte AS
WITH a AS (SELECT player_id, club_familia_id, SUM(peso) AS fifa, SUM(peso - peso_relleno) AS fifa_sin_relleno
           FROM {T}.gold_formacion_fifa GROUP BY player_id, club_familia_id),
     d AS (SELECT player_id, club_familia_id, SUM(peso_norm) AS debut, SUM(peso_norm - peso_relleno_norm) AS debut_sin_relleno
           FROM {T}.gold_formacion_debut GROUP BY player_id, club_familia_id),
     p AS (SELECT COALESCE(a.player_id, d.player_id) AS player_id,
                  COALESCE(a.club_familia_id, d.club_familia_id) AS club_familia_id,
                  COALESCE(a.fifa, 0) AS fifa, COALESCE(a.fifa_sin_relleno, 0) AS fifa_sin_relleno,
                  COALESCE(d.debut, 0) AS debut, COALESCE(d.debut_sin_relleno, 0) AS debut_sin_relleno
           FROM a FULL OUTER JOIN d ON a.player_id = d.player_id AND a.club_familia_id = d.club_familia_id),
     m AS (SELECT player_id,
                  SUM(CASE WHEN rol_club <> 'juvenil' THEN minutos END)                 AS minutos_no_juv,
                  SUM(CASE WHEN rol_club <> 'juvenil' THEN goles END)                   AS goles_no_juv,
                  SUM(CASE WHEN rol_club <> 'juvenil' THEN asistencias END) AS asistencias_no_juv,
                  SUM(CASE WHEN competition_id IN ({GRANDES}) THEN partidos END)        AS partidos_7_grandes,
                  SUM(CASE WHEN competition_id = 'CL' THEN partidos END)                AS partidos_champions
           FROM {T}.gold_minutos GROUP BY player_id),
     -- solo la selección mayor de Argentina
     s AS (SELECT CAST(player_id AS STRING) AS player_id,
                  partidos_mayor_arg AS partidos_mayor, goles_mayor_arg AS goles_mayor
           FROM {T}.gold_seleccion),
     t AS (SELECT player_id, COUNT(*) AS titulos FROM {T}.gold_logros WHERE tipo_logro = 'titulo' GROUP BY player_id),
     v AS (SELECT CAST(player_id AS STRING) AS player_id, SUM(clean_sheets) AS vallas_invictas
           FROM {T}.silver_player_stats GROUP BY CAST(player_id AS STRING)),
     -- datos del club (misma lógica que gold_ranking_formados, pero sobre todos los clubes de gold_formacion_fifa)
     nm AS (SELECT club_familia_id, MAX(club_familia_name) AS club_familia_name
            FROM {T}.gold_formacion_fifa GROUP BY club_familia_id),
     pa AS (SELECT club_id, pais FROM (
              SELECT club_id, pais,
                     ROW_NUMBER() OVER (PARTITION BY club_id ORDER BY n DESC, pais) AS rn
              FROM (
                SELECT club_id, pais, COUNT(*) AS n
                FROM (
                  SELECT CAST(from_club_id AS STRING) AS club_id, from_country_name AS pais FROM {T}.silver_transfers
                  UNION ALL
                  SELECT CAST(to_club_id AS STRING),   to_country_name FROM {T}.silver_transfers
                ) WHERE pais IS NOT NULL
                GROUP BY club_id, pais
              )
            ) WHERE rn = 1),
     c105 AS (SELECT club_id FROM {T}.gold_club_familia WHERE origen = 'argentino_105' AND rol = 'padre'),
     -- camada = año de debut en un club que no es juvenil (misma definición que gold_ranking_camada)
     db AS (SELECT player_id, CAST(temporada_debut AS INT) AS camada FROM {T}.gold_debut)
SELECT p.player_id, p.club_familia_id,
       COALESCE(nm.club_familia_name, CONCAT('Sin nombre (', p.club_familia_id, ')')) AS club_familia_name,
       pa.pais                                  AS pais,
       COALESCE(pa.pais = 'Argentina', false)   AS es_argentino,
       (c105.club_id IS NOT NULL)               AS en_los_105,
       db.camada                                AS camada,
       FLOOR(db.camada / 5) * 5                 AS lustro_desde,
       CASE WHEN db.camada IS NULL THEN 'Sin debut'
            ELSE CONCAT(CAST(FLOOR(db.camada / 5) * 5 AS INT), '-', CAST(FLOOR(db.camada / 5) * 5 + 4 AS INT))
       END                                      AS lustro,
       ROUND(p.fifa, 6) AS fifa, ROUND(p.fifa_sin_relleno, 6) AS fifa_sin_relleno,
       ROUND(p.debut, 6) AS debut, ROUND(p.debut_sin_relleno, 6) AS debut_sin_relleno,
       COALESCE(m.minutos_no_juv, 0)       AS minutos_no_juv,
       COALESCE(m.goles_no_juv, 0)         AS goles_no_juv,
       COALESCE(s.partidos_mayor, 0)       AS partidos_mayor,
       COALESCE(s.goles_mayor, 0)          AS goles_mayor,
       COALESCE(t.titulos, 0)              AS titulos,
       COALESCE(m.partidos_7_grandes, 0)   AS partidos_7_grandes,
       COALESCE(m.partidos_champions, 0)   AS partidos_champions,
       COALESCE(m.asistencias_no_juv, 0) AS asistencias_no_juv,
       COALESCE(v.vallas_invictas, 0)      AS vallas_invictas,
       COALESCE(j.position_group, 'Sin posición') AS position_group,
       COALESCE(j.position_main,  'Sin posición') AS position_main
FROM p
LEFT JOIN nm   ON nm.club_familia_id = p.club_familia_id
LEFT JOIN pa   ON pa.club_id         = p.club_familia_id
LEFT JOIN c105 ON c105.club_id       = p.club_familia_id
LEFT JOIN db   ON db.player_id       = p.player_id
LEFT JOIN m ON m.player_id = p.player_id
LEFT JOIN s ON s.player_id = p.player_id
LEFT JOIN t ON t.player_id = p.player_id
LEFT JOIN v ON v.player_id = p.player_id
LEFT JOIN {T}.gold_jugador j ON j.player_id = p.player_id
""")

# Chequeo 1: suma de pesos por criterio = jugadores con peso; filas
display(spark.sql(f"""SELECT COUNT(*) AS filas, COUNT(DISTINCT player_id) AS jugadores, COUNT(DISTINCT club_familia_id) AS clubes,
  ROUND(SUM(debut), 2) AS suma_debut, ROUND(SUM(fifa), 2) AS suma_fifa FROM {T}.gold_aporte"""))

# Chequeo 2: blancos en los datos del club (esperado: nombre en blanco = 0; los "Sin nombre (id)" son los clubes
# que no tienen nombre en ninguna tabla, y los sin país quedan con es_argentino = false)
display(spark.sql(f"""SELECT
  COUNT(DISTINCT CASE WHEN club_familia_name LIKE 'Sin nombre (%' THEN club_familia_id END) AS clubes_sin_nombre,
  COUNT(DISTINCT CASE WHEN pais IS NULL THEN club_familia_id END)                           AS clubes_sin_pais,
  COUNT(DISTINCT CASE WHEN club_familia_name IS NULL THEN club_familia_id END)              AS nombre_nulo
FROM {T}.gold_aporte"""))

# Chequeo 3 (lustro): jugadores por lustro; la suma tiene que dar los mismos jugadores del chequeo 1
display(spark.sql(f"""SELECT lustro, COUNT(DISTINCT player_id) AS jugadores, COUNT(*) AS filas
FROM {T}.gold_aporte GROUP BY lustro, lustro_desde ORDER BY lustro_desde NULLS LAST"""))

## gold_ingresos
Ingresos de los clubes formadores por los traspasos de sus jugadores. La celda crea la tabla con `CREATE OR REPLACE TABLE`.

Parte de los traspasos de tipo `transfer` con `fee` mayor a 0, con fecha y sin `upcoming`, de jugadores de `gold_jugador`. Para cada uno calcula el peso de solidaridad de cada club familia con la regla de `gold_formacion_fifa`, hasta la fecha del traspaso. Se reparte el 5 % del `fee`.

| `tipo` | `monto` |
|---|---|
| `solidaridad` | 5 % del `fee` por el peso de solidaridad del club |
| `venta_propia` | `fee` menos el 5 % del `fee` por el peso total de solidaridad. Va para la familia del club vendedor |

Columnas: `player_id`, `transfer_id`, `fecha`, `season`, `edad_transfer`, `fee`, `tipo`, `club_familia_id`, `monto`, `peso_solidaridad`, `club_familia_name` y `en_los_105`.

Controles: filas, traspasos y montos por tipo, que la solidaridad por traspaso no supere el 5 % del `fee`, traspasos con `fee` sin solidaridad y el traspaso con más `fee` que tiene solidaridad.

In [0]:
%python
# gold_ingresos
T = "workspace.inferiores"

spark.sql(f"""
CREATE OR REPLACE TABLE {T}.gold_ingresos AS
WITH tf AS (
  SELECT CAST(t.player_id AS STRING) AS player_id, t.transfer_id, CAST(t.date AS DATE) AS fecha,
         t.season, t.age AS edad_transfer, t.fee,
         COALESCE(f.parent_club_id, CAST(t.from_club_id AS STRING)) AS club_vendedor_id
  FROM {T}.silver_transfers t
  LEFT JOIN {T}.gold_club_familia f ON f.club_id = CAST(t.from_club_id AS STRING)
  WHERE t.transfer_type = 'transfer' AND t.fee > 0 AND NOT COALESCE(t.upcoming, false)
    AND t.date IS NOT NULL
    AND CAST(t.player_id AS STRING) IN (SELECT player_id FROM {T}.gold_jugador)
),
nac AS (
  SELECT player_id, MIN(anio - edad_fifa) AS anio_nac
  FROM {T}.gold_formacion_fifa
  WHERE player_id IN (SELECT player_id FROM tf)
  GROUP BY player_id
),
anios AS (
  SELECT player_id, anio_nac, explode(sequence(anio_nac + 12, anio_nac + 23)) AS anio FROM nac
),
tr AS (
  SELECT player_id, club_familia_id AS familia_id, fecha_desde, fecha_hasta
  FROM {T}.gold_linea_tiempo WHERE rol_club <> 'no_club'
),
cruce AS (
  SELECT x.transfer_id, a.anio, a.anio - a.anio_nac AS edad_fifa, t.familia_id,
         GREATEST(0, DATEDIFF(
           LEAST(COALESCE(t.fecha_hasta, current_date()), make_date(a.anio + 1, 1, 1), x.fecha),
           GREATEST(COALESCE(t.fecha_desde, make_date(a.anio_nac + 12, 1, 1)), make_date(a.anio, 1, 1))
         )) AS dias
  FROM tf x
  JOIN anios a ON a.player_id = x.player_id
  JOIN tr t ON t.player_id = x.player_id
),
sol AS (
  SELECT transfer_id, familia_id AS club_familia_id,
         SUM((CASE WHEN edad_fifa <= 15 THEN 0.05 ELSE 0.10 END) * dias
             / DATEDIFF(make_date(anio + 1, 1, 1), make_date(anio, 1, 1))) AS peso_solidaridad
  FROM cruce WHERE dias > 0
  GROUP BY transfer_id, familia_id
),
r AS (
  SELECT x.player_id, x.transfer_id, x.fecha, x.season, x.edad_transfer, x.fee,
         'venta_propia' AS tipo, x.club_vendedor_id AS club_familia_id,
         CAST(x.fee - 0.05 * x.fee * COALESCE(st.peso, 0) AS DOUBLE) AS monto, CAST(NULL AS DOUBLE) AS peso_solidaridad
  FROM tf x
  LEFT JOIN (SELECT transfer_id, SUM(peso_solidaridad) AS peso FROM sol GROUP BY transfer_id) st
  ON st.transfer_id = x.transfer_id
  UNION ALL
  SELECT x.player_id, x.transfer_id, x.fecha, x.season, x.edad_transfer, x.fee,
         'solidaridad', s.club_familia_id, 0.05 * x.fee * s.peso_solidaridad, s.peso_solidaridad
  FROM sol s JOIN tf x ON x.transfer_id = s.transfer_id
),
nom AS (SELECT club_id, MAX(club_name) AS club_name FROM {T}.gold_linea_tiempo GROUP BY club_id)
SELECT r.*, COALESCE(f.club_name, n.club_name) AS club_familia_name,
       COALESCE(f.origen = 'argentino_105' AND f.rol = 'padre', false) AS en_los_105
FROM r
LEFT JOIN {T}.gold_club_familia f ON f.club_id = r.club_familia_id
LEFT JOIN nom n ON n.club_id = r.club_familia_id
""")

# A: filas, traspasos y montos por tipo
display(spark.sql(f"""SELECT tipo, COUNT(*) AS filas, COUNT(DISTINCT transfer_id) AS traspasos,
  COUNT(DISTINCT club_familia_id) AS clubes, ROUND(SUM(monto)) AS monto_total FROM {T}.gold_ingresos GROUP BY tipo"""))
# B: la solidaridad repartida por traspaso nunca puede superar el 5 % del fee (esperado: 0 filas)
display(spark.sql(f"""SELECT transfer_id, MAX(fee) AS fee, SUM(monto) AS solidaridad
  FROM {T}.gold_ingresos WHERE tipo = 'solidaridad' GROUP BY transfer_id
  HAVING SUM(monto) > 0.0501 * MAX(fee)"""))
# C: traspasos con fee que no generaron ninguna fila de solidaridad (jugadores sin línea de tiempo o sin nacimiento)
display(spark.sql(f"""SELECT COUNT(DISTINCT transfer_id) AS sin_solidaridad FROM {T}.gold_ingresos
  WHERE tipo = 'venta_propia' AND transfer_id NOT IN (SELECT transfer_id FROM {T}.gold_ingresos WHERE tipo = 'solidaridad')"""))
# D: un caso para mirar a mano
display(spark.sql(f"""SELECT * FROM {T}.gold_ingresos WHERE transfer_id IN (
  SELECT transfer_id FROM {T}.gold_ingresos WHERE tipo = 'solidaridad' ORDER BY fee DESC LIMIT 1) ORDER BY tipo, monto DESC"""))

## Cierre del ciclo
`marcar("publicado_en")` guarda la hora actual en `bronze_ciclos` para el ciclo vigente, solo si `cerrado_en` ya tiene valor y `publicado_en` estaba vacío.

In [0]:
marcar("publicado_en")